# 05 · Avaliação

**CRISP-DM fase 5 — avaliação** · Meridiano Imobiliária

---

O notebook [`04-modelagem`](04-modelagem.ipynb) entregou um modelo: **K-Means com *k* = 3**,
silhueta 0,1726, escolhido pela regra declarada antes do resultado — *o maior k que passa em
todos os critérios*. Entregou também um critério **pendente**, que ele não podia medir sem
quebrar a própria regra: o ARI da segmentação contra os quartis de `preco_venda`.

Esta fase faz outra pergunta. A fase 4 perguntou se o modelo é **tecnicamente bom**; a fase 5
pergunta se ele **resolve o problema que a diretoria trouxe** — e se o caminho até ele suporta
ser auditado. São perguntas diferentes, e um modelo pode passar na primeira e reprovar na
segunda.

| Bloco | Tarefa do CRISP-DM | Pergunta | O que decide |
|:--|:---|:---|:---|
| 1 | *(reexame da fase 4)* | A fase 4 deixou silhueta na mesa? | A varredura ampla — 126 configurações de *k*, algoritmo, escala e dimensão |
| 2 | *Evaluate results* | Os segmentos servem ao objetivo de negócio? | O rótulo reservado, aberto pela primeira vez, contra os critérios do Canvas |
| 3 | *Review process* | O caminho até aqui aguenta auditoria? | A cadeia refeita, a conferência de vazamento e as divergências encontradas |
| 4 | *Determine next steps* | Implanta, revisa ou para? | A decisão de seguir, com o que fica pendente escrito |

## O bloco 1 e o que o pedido significa

A tarefa era **testar várias mudanças de *k* para encontrar a melhor silhueta**. O bloco 1 faz
isso literalmente, e vai além do que a fase 4 varreu: 126 configurações, cruzando *k* de 2 a 10
com dez algoritmos, quatro escalonadores e quatro dimensionalidades.

O resultado é o argumento central deste notebook, e vale dizê-lo antes de aparecer:

> **A melhor silhueta do projeto separa 1 imóvel dos outros 2.349.**

Encontrar a melhor silhueta e escolher a melhor segmentação são coisas diferentes, e o bloco 1
mostra a distância entre as duas com três medidas, não com uma afirmação. É por isso que a
varredura pertence à fase 5: ela não escolhe *k*, ela **audita a escolha** que a fase 4 fez —
que é exatamente o que *review process* quer dizer.

## O que muda aqui: o rótulo abre

Até este notebook, `preco_venda`, `preco_por_pe2` e `bairro` estiveram fora de todo arquivo que
alimentou algoritmo. A fase 3 os reservou fisicamente; a fase 4 conferiu duas vezes que não
haviam entrado.

Aqui eles entram — e o momento é declarado, porque é o único em que abrir o rótulo é legítimo:
os segmentos **já estão definidos** e nenhum número deste notebook volta para a modelagem. O
`join` por `pid` do bloco 0 é esse momento, e ele acontece uma vez só.

In [ ]:
# Importações
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import load
from scipy.stats import chi2_contingency
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.model import distancias as dist
from src.model import particional as part
from src.model import perfilagem as perf
from src.model import personas as pers
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

io.configurar_estilo()

# As figuras deste notebook vão para `docs/imagens/avaliacao/`; as tabelas, para
# `reports/tables/avaliacao/`. A regra do projeto é que nenhuma figura existe sem
# a tabela que a produziu.
FIGURAS = config.DOCS / "imagens" / "avaliacao"
FIGURAS.mkdir(parents=True, exist_ok=True)
SECAO = "avaliacao"

# Cores da identidade do projeto (docs/assets/css/brand.css).
BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
DESTAQUE = config.AZUL_CLARO
ALERTA = config.CORAL
NEUTRO = config.CINZA
APROVADO = config.VERDE


def salvar(fig, nome, dados=None):
    """Grava a figura em `docs/imagens/avaliacao/` e a tabela que a originou."""
    fig.savefig(FIGURAS / f"{nome}.png", bbox_inches="tight")
    plt.close(fig)
    if dados is not None:
        io.salvar_tabela(dados, SECAO, nome, indice=False)


print(f"Projeto em {RAIZ}")
print(f"Semente do projeto: {config.SEMENTE}")

## O registro de julgamentos

A fase 4 registrou *decisões de modelagem*. Esta registra **julgamentos**, e a diferença não é
de forma: um julgamento pode **confirmar** ou **derrubar** uma decisão de fase anterior, e o
registro precisa deixar visível qual dos dois aconteceu.

| `acao` | Significado |
|:---|:---|
| `critério medido` | Um critério do Canvas confrontado com o número que o atende ou não |
| `decisão mantida` | A fase 4 decidiu, esta fase reexaminou e a decisão fica de pé — com a medida nova ao lado |
| `decisão revista` | A fase 4 decidiu e esta fase derruba, com o número que derrubou |
| `armadilha registrada` | Um resultado que *parece* melhor e não é, com a medida que expõe o engano |
| `divergência registrada` | O que o repositório afirma e o que a medida diz — conflito aberto, não resolvido em silêncio |
| `próxima etapa` | O que a fase 6 recebe como tarefa, com o critério que a fecha |

A distinção que mais importa numa banca é entre `decisão mantida` e `armadilha registrada`. A
primeira diz que a fase 4 acertou; a segunda diz **por que** a alternativa que parecia melhor
não era — e é essa segunda que separa uma avaliação de uma repetição do que já foi dito.

In [ ]:
JULGAMENTOS: list[dict] = []


def pt(numero) -> str:
    """Formata o número no padrão brasileiro: ponto como separador de milhar."""
    return f"{numero:,}".replace(",", ".")


def usd(valor) -> str:
    """Formata um valor monetário da base, que está em dólares."""
    return f"US$ {pt(int(round(valor)))}"


def registrar(
    bloco: str,
    julgamento: str,
    objeto: str,
    motivo: str,
    evidencia: str,
    acao: str = "critério medido",
):
    """Anota um julgamento da fase 5, com o objeto julgado, o motivo e a medida.

    `evidencia` guarda o **número** que sustenta o julgamento. Avaliação sem
    número ao lado é opinião, e opinião não sobrevive a uma reexecução.
    """
    JULGAMENTOS.append(
        {
            "n": len(JULGAMENTOS) + 1,
            "bloco": bloco,
            "julgamento": julgamento,
            "objeto": objeto,
            "motivo": motivo,
            "evidencia": evidencia,
            "acao": acao,
        }
    )
    print(f"{len(JULGAMENTOS):>2}. [{acao:>22}] {julgamento} — {evidencia}")

# 0 · A entrada — e o momento em que o rótulo abre

Cinco arquivos entram aqui, e um deles não entrava num notebook desde a fase 2:

| Arquivo | Papel |
|:---|:---|
| `data/processed/base_modelagem.parquet` | O espaço em que os segmentos foram formados — sem rótulo |
| `data/processed/base_segmentada.parquet` | O resultado da fase 4: `pid`, segmento, silhueta individual |
| `models/modelo-segmentacao.joblib` | Receita e agrupador ajustados, com o mapa canônico |
| `data/processed/manifesto_modelagem.json` | O que a fase 4 decidiu, com que medida |
| `data/processed/imoveis_curados.parquet` | **A base com `preco_venda` e `bairro`** |

O `join` entre as duas últimas linhas é o momento declarado de abertura do rótulo. Ele é feito
por `pid`, conferido como **um-para-um** e em **ordem preservada** — se a ordem escorregasse,
todo número de negócio deste notebook estaria associado ao imóvel errado, e nada na saída
denunciaria isso.

Três conferências antes de qualquer avaliação:

1. **o modelo gravado reproduz a partição avaliada.** Se `modelo-segmentacao.joblib` devolver
   rótulos diferentes de `base_segmentada.parquet`, o que a fase 4 avaliou não é o que a fase 6
   vai implantar — e a avaliação inteira seria sobre um objeto que não existe;
2. **a base de modelagem continua sem o rótulo.** Redundante com a fase 4 de propósito: as duas
   pontas do vazamento têm de falhar ao mesmo tempo para o erro passar;
3. **o `join` não perde nem duplica imóvel.** 2.350 entram, 2.350 saem, com os mesmos `pid`.

In [ ]:
BASE_MODELAGEM = config.DATA_PROCESSED / "base_modelagem.parquet"
BASE_SEGMENTADA = config.DATA_PROCESSED / "base_segmentada.parquet"
MODELO = config.MODELS / "modelo-segmentacao.joblib"

base = pd.read_parquet(BASE_MODELAGEM)
segmentada = pd.read_parquet(BASE_SEGMENTADA)
manifesto_preparacao = io.ler_json(config.DATA_PROCESSED / "manifesto_preparacao.json")
manifesto_modelagem = io.ler_json(config.DATA_PROCESSED / "manifesto_modelagem.json")
pacote = load(MODELO)

ESPACO_EUCLIDIANO = manifesto_preparacao["espaco_euclidiano"]
ESPACO_MISTO = manifesto_preparacao["espaco_misto"]
LINHAS = len(base)
K_ESCOLHIDO = int(manifesto_modelagem["k"])
ALGORITMO = manifesto_modelagem["algoritmo"]
SEMENTE = config.SEMENTE

PROIBIDAS = {"preco_venda", "preco_por_pe2", "bairro"}
assert not PROIBIDAS & set(base.columns), "rótulo ou reservada entrou na base de modelagem"
assert not PROIBIDAS & set(segmentada.columns), "rótulo entrou na base segmentada"
assert not PROIBIDAS & set(manifesto_modelagem["variaveis"]), "o modelo foi ajustado no rótulo"
assert (segmentada.pid.to_numpy() == base.pid.to_numpy()).all(), "segmentada fora de ordem"

# --------------------------------------------------------------------------- #
# O momento declarado: o rótulo reservado entra, e entra uma vez só.
# --------------------------------------------------------------------------- #
curada = pd.read_parquet(config.BASE_CURADA).set_index("pid").loc[base.pid].reset_index()

assert len(curada) == LINHAS, "o join por pid perdeu ou duplicou imóvel"
assert (curada.pid.to_numpy() == base.pid.to_numpy()).all(), "o join por pid saiu de ordem"
assert curada.preco_venda.notna().all(), "há imóvel sem preço na base curada"

ROTULOS = segmentada.segmento.to_numpy()
RECEITA_TOTAL = float(curada.preco_venda.sum())

entrada = pd.Series(
    {
        "imóveis": pt(LINHAS),
        "modelo da fase 4": f"{ALGORITMO}, k = {K_ESCOLHIDO}",
        "silhueta declarada": f"{manifesto_modelagem['silhueta']:.4f}",
        "espaço": f"{len(ESPACO_EUCLIDIANO)} variáveis, escala padrão",
        "rótulo aberto agora": ", ".join(sorted(PROIBIDAS)),
        "receita da carteira": usd(RECEITA_TOTAL),
        "critério pendente": manifesto_modelagem["criterio_pendente"],
    },
    name="entrada da fase 5",
)
entrada.to_frame()

In [ ]:
# A receita é reajustada aqui para reconstruir o espaço euclidiano. É a mesma
# operação que a fase 4 fez: o arquivo guarda unidades de negócio, e o espaço é
# reconstruído a partir da receita, nunca lido pronto do disco.
preprocessador = load(config.MODELS / "preprocessador-euclidiano.joblib")
assert not hasattr(preprocessador, "transformers_"), "o pré-processador chegou já ajustado"

EUCLIDIANO = preprocessador.fit_transform(base[ESPACO_EUCLIDIANO])
GOWER = dist.gower(base, ESPACO_EUCLIDIANO, ESPACO_MISTO)

# Conferência 1: o modelo gravado reproduz exatamente a partição que foi avaliada.
crus = pacote["pipeline"].predict(base[ESPACO_EUCLIDIANO])
mapa = {int(cru): int(segmento) for cru, segmento in pacote["mapa_canonico"].items()}
reproduzidos = np.array([mapa[int(cru)] for cru in crus])

REPRODUZ = bool((reproduzidos == ROTULOS).all())
ARI_REPRODUCAO = round(float(adjusted_rand_score(reproduzidos, ROTULOS)), 4)
assert REPRODUZ, "o modelo gravado não reproduz a partição avaliada na fase 4"

SILHUETA_ESCOLHIDO = round(float(silhouette_score(EUCLIDIANO, ROTULOS)), 4)

print(f"Espaço euclidiano: {EUCLIDIANO.shape[0]} × {EUCLIDIANO.shape[1]}")
print(f"Matriz de Gower:   {GOWER.shape[0]} × {GOWER.shape[1]}")
print(f"O modelo gravado reproduz a partição avaliada: {REPRODUZ} (ARI {ARI_REPRODUCAO:.4f})")
print(f"Silhueta recalculada: {SILHUETA_ESCOLHIDO:.4f} "
      f"(manifesto da fase 4: {manifesto_modelagem['silhueta']:.4f})")

# 1 · A varredura ampla de *k* — a busca pela melhor silhueta

A tarefa é direta: **variar *k* e encontrar a melhor silhueta**. A fase 4 varreu *k* de 2 a 10
com o K-Means e comparou cinco algoritmos num único *k*. Este bloco varre o produto cartesiano
completo, que é o que responde à pergunta sem depender do recorte que a fase 4 escolheu.

| Eixo da varredura | Valores | Por que este eixo |
|:---|:---|:---|
| **Número de grupos** | *k* de 2 a 10 | O eixo pedido — inclui a faixa fora do teto de negócio, para ver se o teto corta estrutura |
| **Algoritmo** | K-Means, hierárquica (`ward`, `average`, `complete`, `single`), GMM (`full`, `tied`, `diag`, `spherical`), K-Medoids sobre Gower | Dez famílias com objetivos diferentes; a silhueta não é o objetivo de nenhuma delas |
| **Escala** | `padrao`, `minmax`, `robusto`, `sem_escala` | A silhueta é uma razão entre distâncias, e distância depende da escala |
| **Dimensão** | 17 variáveis e PCA com 8, 4 e 2 componentes | A silhueta é calculada no espaço em que se agrupa, e espaço menor comprime distância |

São **126 configurações**. A varredura é grande de propósito: o objetivo não é achar um número
maior do que 0,1726, é descobrir **o que precisa ser feito** para achá-lo — e se o que precisa
ser feito é aceitável.

## 1.1 · A grade principal — *k* × algoritmo

O primeiro corte mantém escala e dimensão fixas no que a fase 3 decidiu (escala padrão, 17
variáveis) e varre apenas *k* e algoritmo. Duas curvas por configuração: a silhueta, que é o
que se procura, e o **tamanho do menor grupo**, que é o que a silhueta não vê.

In [ ]:
def rotular(algoritmo: str, k: int, dados: np.ndarray) -> np.ndarray:
    """Devolve os rótulos de um algoritmo para um k, no espaço informado."""
    if algoritmo == "K-Means":
        return KMeans(n_clusters=k, init="k-means++", n_init=20,
                      random_state=SEMENTE).fit_predict(dados)
    if algoritmo.startswith("Hierárquica"):
        ligacao = algoritmo.split("(")[1].rstrip(")")
        return AgglomerativeClustering(n_clusters=k, linkage=ligacao).fit_predict(dados)
    if algoritmo.startswith("GMM"):
        covariancia = algoritmo.split("(")[1].rstrip(")")
        return GaussianMixture(n_components=k, covariance_type=covariancia, n_init=5,
                               max_iter=500, random_state=SEMENTE).fit_predict(dados)
    raise ValueError(f"algoritmo desconhecido: {algoritmo}")


ALGORITMOS = (
    ["K-Means"]
    + [f"Hierárquica ({ligacao})" for ligacao in ("ward", "average", "complete", "single")]
    + [f"GMM ({covariancia})" for covariancia in ("full", "tied", "diag", "spherical")]
)
K_VARREDURA = range(2, 11)

linhas = []
for k in K_VARREDURA:
    for algoritmo in ALGORITMOS:
        rotulos = rotular(algoritmo, k, EUCLIDIANO)
        medidas = part.metricas_internas(EUCLIDIANO, rotulos)
        contagem = np.bincount(rotulos)
        linhas.append(
            {
                "algoritmo": algoritmo,
                "espaco": "euclidiano (17)",
                "k": k,
                "silhueta": round(medidas["silhueta"], 4),
                "davies_bouldin": round(medidas["davies_bouldin"], 3),
                "menor_grupo": int(contagem.min()),
                "menor_grupo_pct": round(100 * contagem.min() / LINHAS, 3),
                "maior_grupo_pct": round(100 * contagem.max() / LINHAS, 1),
            }
        )
    # O K-Medoids opera sobre a dissimilaridade de Gower: espaço próprio, e a
    # silhueta dele **não** é comparável com a das linhas acima. Fica marcado.
    resultado = part.kmedoids(GOWER, k, reinicios=8, semente=SEMENTE)
    contagem = np.bincount(resultado.rotulos)
    linhas.append(
        {
            "algoritmo": "K-Medoids (Gower)",
            "espaco": "Gower (22)",
            "k": k,
            "silhueta": round(
                part.metricas_internas(None, resultado.rotulos, GOWER)["silhueta"], 4
            ),
            "davies_bouldin": np.nan,
            "menor_grupo": int(contagem.min()),
            "menor_grupo_pct": round(100 * contagem.min() / LINHAS, 3),
            "maior_grupo_pct": round(100 * contagem.max() / LINHAS, 1),
        }
    )

grade = pd.DataFrame(linhas)
print(f"{len(grade)} configurações de k × algoritmo avaliadas")

In [ ]:
# Os critérios NÃO são redefinidos aqui: são lidos do manifesto que a fase 4
# gravou. Reescrevê-los neste notebook permitiria ajustá-los ao resultado, que é
# exatamente o que a fase 4 se proibiu de fazer.
CRITERIOS = manifesto_modelagem["criterios"]
SILHUETA_MINIMA = CRITERIOS["silhueta_minima"]
MENOR_SEGMENTO_MINIMO = CRITERIOS["menor_segmento_minimo"]
K_MINIMO, K_MAXIMO = CRITERIOS["k_minimo"], CRITERIOS["k_maximo"]
ARI_MINIMO = CRITERIOS["ari_minimo"]

fig, eixos = plt.subplots(1, 2, figsize=(16, 5.6), constrained_layout=True)

cores_algoritmo = dict(
    zip(
        ALGORITMOS + ["K-Medoids (Gower)"],
        list(config.PALETA) + [config.CINZA, config.AZUL_COBALTO],
    )
)
for algoritmo, recorte in grade.groupby("algoritmo", sort=False):
    destacado = algoritmo in ("K-Means", "Hierárquica (single)")
    eixos[0].plot(recorte.k, recorte.silhueta, marker="o", ms=4,
                  lw=2.6 if destacado else 1.3,
                  ls="dashed" if algoritmo == "K-Medoids (Gower)" else "solid",
                  color=cores_algoritmo[algoritmo], label=algoritmo,
                  alpha=1.0 if destacado else 0.7)
eixos[0].axhline(SILHUETA_MINIMA, color=REFERENCIA, ls="dotted", lw=2)
eixos[0].text(10, SILHUETA_MINIMA, f" piso do Canvas: {SILHUETA_MINIMA:.2f} ", va="bottom",
              ha="right", fontsize=9, color=config.AZUL_PROFUNDO)
eixos[0].scatter([K_ESCOLHIDO], [SILHUETA_ESCOLHIDO], s=200, marker="*", zorder=5,
                 color=ALERTA, edgecolor="white", linewidth=1.2)
eixos[0].annotate(f"fase 4: k = {K_ESCOLHIDO}\n{SILHUETA_ESCOLHIDO:.4f}",
                  (K_ESCOLHIDO, SILHUETA_ESCOLHIDO), textcoords="offset points",
                  xytext=(16, -52), fontsize=9, color=ALERTA, fontweight="bold")
eixos[0].set_xticks(list(K_VARREDURA))
eixos[0].set_xlabel("k")
eixos[0].set_ylabel("Silhueta")
eixos[0].set_title("A silhueta em 90 configurações — o topo não está onde se procura")
eixos[0].legend(fontsize=7.5, ncols=2, loc="upper right")

for algoritmo, recorte in grade.groupby("algoritmo", sort=False):
    destacado = algoritmo in ("K-Means", "Hierárquica (single)")
    eixos[1].plot(recorte.k, recorte.menor_grupo_pct, marker="o", ms=4,
                  lw=2.6 if destacado else 1.3, color=cores_algoritmo[algoritmo],
                  alpha=1.0 if destacado else 0.65)
eixos[1].axhline(100 * MENOR_SEGMENTO_MINIMO, color=REFERENCIA, ls="dotted", lw=2)
eixos[1].text(10, 100 * MENOR_SEGMENTO_MINIMO,
              f" piso do Canvas: {100 * MENOR_SEGMENTO_MINIMO:.0f}% da carteira ", va="bottom",
              ha="right", fontsize=9, color=config.AZUL_PROFUNDO)
eixos[1].set_yscale("log")
eixos[1].set_xticks(list(K_VARREDURA))
eixos[1].set_xlabel("k")
eixos[1].set_ylabel("Menor grupo (% da carteira — escala log)")
eixos[1].set_title("O que a silhueta não vê — o tamanho do menor grupo")

salvar(fig, "varredura-ampla-de-k", grade)
grade.sort_values("silhueta", ascending=False).head(8)

### 1.2 · O campeão literal, e o que ele entrega

A configuração de maior silhueta em toda a grade é a **hierárquica com ligação `single`**, e
ela ocupa as primeiras posições do ranking com *k* = 2, 3 e 4. A silhueta chega perto de
**0,32** — quase o dobro dos 0,1726 do modelo escolhido pela fase 4.

O painel da direita mostra o que ela cobra por isso, e o eixo está em escala logarítmica porque
em escala linear o valor não seria visível: o menor grupo tem **0,04% da carteira**. Um imóvel.
A segmentação campeã em silhueta divide a Meridiano em *um imóvel* e *os outros 2.349*.

Não é defeito de implementação, é o mecanismo da ligação `single` funcionando como projetado:
ela funde pelo par mais próximo, o que produz **encadeamento** — um grupo absorve a carteira
inteira enquanto os demais ficam com pontos isolados na periferia. A silhueta premia isso
porque o imóvel isolado está, de fato, longe do grupo vizinho e perto de si mesmo. A métrica
está certa; a pergunta é que estava errada.

In [ ]:
campeao = grade.sort_values("silhueta", ascending=False).iloc[0]
SILHUETA_MAXIMA = float(campeao.silhueta)
CAMPEAO_LITERAL = f"{campeao.algoritmo}, k = {int(campeao.k)}"
MENOR_GRUPO_CAMPEAO = int(campeao.menor_grupo)

topo = grade.sort_values("silhueta", ascending=False).head(10).copy()
topo["ganho_sobre_a_fase_4"] = (topo.silhueta - SILHUETA_ESCOLHIDO).round(4)
topo["admissivel"] = (
    (topo.menor_grupo_pct >= 100 * MENOR_SEGMENTO_MINIMO)
    & (topo.k >= K_MINIMO)
    & (topo.k <= K_MAXIMO)
    & (topo.espaco == "euclidiano (17)")
)

fig, ax = plt.subplots(figsize=(13, 5.8))
rotulos_y = [f"{a} · k={k}" for a, k in zip(topo.algoritmo, topo.k)]
ax.barh(rotulos_y, topo.silhueta,
        color=[APROVADO if ok else ALERTA for ok in topo.admissivel])
ax.bar_label(
    ax.containers[0],
    labels=[f"{s:.4f}    menor grupo: {pt(n)} imóve{'is' if n > 1 else 'l'} ({p:.2f}%)"
            for s, n, p in zip(topo.silhueta, topo.menor_grupo, topo.menor_grupo_pct)],
    padding=4, fontsize=8.5,
)
ax.axvline(SILHUETA_ESCOLHIDO, color=BARRA, ls="dashed", lw=2)
ax.text(SILHUETA_ESCOLHIDO, -0.8, f" modelo da fase 4: {SILHUETA_ESCOLHIDO:.4f}",
        color=BARRA, fontsize=9, fontweight="bold", va="bottom")
ax.invert_yaxis()
ax.set_xlim(0, topo.silhueta.max() * 1.6)
ax.set_xlabel("Silhueta")
ax.set_title("As dez maiores silhuetas da grade — verde é admissível, coral não é")

salvar(fig, "as-dez-maiores-silhuetas", topo)
topo[["algoritmo", "k", "silhueta", "ganho_sobre_a_fase_4", "menor_grupo",
      "menor_grupo_pct", "maior_grupo_pct", "admissivel"]]

In [ ]:
registrar(
    "1 · Varredura",
    "A maior silhueta do projeto é inadmissível",
    CAMPEAO_LITERAL,
    "A ligação `single` funde pelo par mais próximo e produz encadeamento: um grupo absorve a "
    "carteira e os demais ficam com pontos isolados. A silhueta premia o isolamento — que é "
    "exatamente o que ela mede, e exatamente o que o Canvas proíbe como segmento.",
    f"silhueta {SILHUETA_MAXIMA:.4f} (+{SILHUETA_MAXIMA - SILHUETA_ESCOLHIDO:.4f} sobre o "
    f"modelo escolhido) com menor grupo de {MENOR_GRUPO_CAMPEAO} imóvel "
    f"({100 * MENOR_GRUPO_CAMPEAO / LINHAS:.3f}% da carteira)",
    acao="armadilha registrada",
)

## 1.3 · As outras duas formas de comprar silhueta

O campeão literal é o caso fácil: qualquer um vê que um grupo de um imóvel não é um segmento.
As duas armadilhas seguintes são mais perigosas porque o resultado **não parece degenerado** —
os grupos ficam com tamanhos razoáveis, e só a comparação com a partição de referência revela o
que aconteceu.

### Armadilha 1 · Trocar a escala

A silhueta é uma razão entre distâncias, e distância depende de quanto cada variável pesa. A
fase 3 escolheu a escala padrão para que 17 variáveis com unidades diferentes — pés quadrados,
anos, notas de 1 a 10 — pesassem igual.

Se o critério fosse a silhueta, a escolha seria outra. E a coluna que desmonta o ganho não é a
da silhueta: é a do **ARI contra a partição de referência**, que responde se o resultado é a
mesma segmentação medida de outro jeito ou uma segmentação diferente.

In [ ]:
ESCALADORES = {
    "padrao": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robusto": RobustScaler(),
    "sem_escala": None,
}

# A partição de referência em cada k: o K-Means no espaço que a fase 3 declarou.
referencia_por_k = {
    k: KMeans(n_clusters=k, n_init=20, random_state=SEMENTE).fit_predict(EUCLIDIANO)
    for k in range(2, 7)
}

linhas_escala = []
for nome, escalador in ESCALADORES.items():
    matriz = (
        base[ESPACO_EUCLIDIANO].to_numpy(dtype=float)
        if escalador is None
        else escalador.fit_transform(base[ESPACO_EUCLIDIANO])
    )
    for k in range(2, 7):
        rotulos = KMeans(n_clusters=k, n_init=20, random_state=SEMENTE).fit_predict(matriz)
        contagem = np.bincount(rotulos)
        linhas_escala.append(
            {
                "escalador": nome,
                "k": k,
                "silhueta": round(float(silhouette_score(matriz, rotulos)), 4),
                "menor_grupo_pct": round(100 * contagem.min() / LINHAS, 2),
                "ari_contra_a_escala_padrao": round(
                    float(adjusted_rand_score(referencia_por_k[k], rotulos)), 3
                ),
            }
        )
escala = pd.DataFrame(linhas_escala)

# Sem escala, quem manda é a variável de maior variância bruta.
variancias = base[ESPACO_EUCLIDIANO].var().sort_values(ascending=False)
DOMINANTE = str(variancias.index[0])
PESO_DOMINANTE = round(100 * variancias.iloc[0] / variancias.sum(), 2)

sem_escala = escala.query("escalador == 'sem_escala' and k == @K_ESCOLHIDO").iloc[0]
SILHUETA_SEM_ESCALA = float(sem_escala.silhueta)
ARI_SEM_ESCALA = float(sem_escala.ari_contra_a_escala_padrao)

print(f"Sem escala, `{DOMINANTE}` responde por {PESO_DOMINANTE:.2f}% da variância do espaço.")
escala.pivot(index="k", columns="escalador", values="silhueta")

### Armadilha 2 · Trocar a dimensão

A fase 3 decidiu **não agrupar sobre componentes principais**, e o motivo registrado foi de
negócio: um segmento descrito como "alto em PC1" não vira linha de produto.

A medida abaixo mostra que existia um motivo adicional, e ele é métrico. Agrupar sobre 2
componentes e medir a silhueta **ali** devolve um número muito maior. Mas a partição é
praticamente a mesma — o ARI contra o K-Means de 17 dimensões fica perto de 1. O que mudou não
foi o agrupamento, foi a régua: projetar em menos dimensões descarta justamente a variação que
tornava os grupos ambíguos, e a silhueta sobe **sem que nenhum imóvel tenha mudado de grupo**.

É a demonstração mais limpa possível de que a silhueta é propriedade do **espaço**, e não da
partição.

In [ ]:
linhas_dimensao = []
for componentes in (17, 8, 4, 2):
    if componentes == len(ESPACO_EUCLIDIANO):
        projetado, variancia_retida = EUCLIDIANO, 100.0
    else:
        projecao = PCA(n_components=componentes, random_state=SEMENTE)
        projetado = projecao.fit_transform(EUCLIDIANO)
        variancia_retida = round(100 * projecao.explained_variance_ratio_.sum(), 1)
    for k in (2, 3, 4, 5):
        rotulos = KMeans(n_clusters=k, n_init=20, random_state=SEMENTE).fit_predict(projetado)
        linhas_dimensao.append(
            {
                "dimensoes": componentes,
                "variancia_retida_pct": variancia_retida,
                "k": k,
                "silhueta_no_espaco_do_ajuste": round(
                    float(silhouette_score(projetado, rotulos)), 4
                ),
                # A MESMA partição, medida no espaço que a fase 3 declarou.
                "silhueta_no_espaco_de_17": round(float(silhouette_score(EUCLIDIANO, rotulos)), 4),
                "ari_contra_o_espaco_de_17": round(
                    float(adjusted_rand_score(referencia_por_k[k], rotulos)), 3
                ),
            }
        )
reducao = pd.DataFrame(linhas_dimensao)

duas_dimensoes = reducao.query("dimensoes == 2 and k == 2").iloc[0]
SILHUETA_PCA2 = float(duas_dimensoes.silhueta_no_espaco_do_ajuste)
ARI_PCA2 = float(duas_dimensoes.ari_contra_o_espaco_de_17)

fig, eixos = plt.subplots(1, 2, figsize=(16, 5.4), constrained_layout=True)

# Os deslocamentos são fixados por escalador: sem isso, as anotações de `minmax`
# e `robusto` se sobrepõem no mesmo k.
DESLOCAMENTO_DA_ANOTACAO = {"minmax": (8, 8), "robusto": (8, -16), "sem_escala": (8, 8)}
for cor, (nome, recorte) in zip(config.PALETA, escala.groupby("escalador", sort=False)):
    eixos[0].plot(recorte.k, recorte.silhueta, marker="o", lw=2.2, color=cor, label=nome)
    if nome != "padrao":
        pico = recorte.loc[recorte.silhueta.idxmax()]
        eixos[0].annotate(f"ARI {pico.ari_contra_a_escala_padrao:.2f}", (pico.k, pico.silhueta),
                          textcoords="offset points", xytext=DESLOCAMENTO_DA_ANOTACAO[nome],
                          fontsize=8.5, color=cor, fontweight="bold")
eixos[0].axhline(SILHUETA_MINIMA, color=REFERENCIA, ls="dotted", lw=2)
eixos[0].set_xticks(range(2, 7))
eixos[0].set_xlabel("k")
eixos[0].set_ylabel("Silhueta")
eixos[0].set_title(f"Trocar a escala compra silhueta — e entrega outra partição\n"
                   f"(sem escala, {dic.rotulo(DOMINANTE)} vale {PESO_DOMINANTE:.0f}% da variância)")
eixos[0].legend(title="escalador", fontsize=8.5)

largura, posicoes = 0.2, np.arange(4)
for deslocamento, (dimensoes, recorte) in zip((-1.5, -0.5, 0.5, 1.5),
                                              reducao.groupby("dimensoes", sort=False)):
    eixos[1].bar(posicoes + deslocamento * largura, recorte.silhueta_no_espaco_do_ajuste,
                 width=largura, label=f"{dimensoes} dimensões",
                 color=config.PALETA[int(1 + deslocamento + 1.5) % len(config.PALETA)])
    for posicao, (_, linha) in zip(posicoes, recorte.iterrows()):
        if dimensoes == 2:
            eixos[1].annotate(
                f"ARI {linha.ari_contra_o_espaco_de_17:.2f}",
                (posicao + deslocamento * largura, linha.silhueta_no_espaco_do_ajuste),
                ha="center", textcoords="offset points", xytext=(0, 4),
                fontsize=8, fontweight="bold", color=config.AZUL_PROFUNDO,
            )
eixos[1].set_xticks(posicoes, [f"k = {k}" for k in (2, 3, 4, 5)])
eixos[1].set_ylabel("Silhueta medida no espaço em que se agrupou")
eixos[1].set_title("Reduzir a dimensão compra silhueta — sem mudar quem está com quem")
eixos[1].legend(fontsize=8.5, ncols=2)
eixos[1].margins(y=0.18)

salvar(fig, "armadilhas-da-silhueta")
io.salvar_tabela(escala, SECAO, "sensibilidade-de-escala", indice=False)
io.salvar_tabela(reducao, SECAO, "sensibilidade-de-dimensao", indice=False)
reducao

### O que as três armadilhas provam

As três formas de melhorar a silhueta encontradas até aqui têm a mesma estrutura, e ela precisa
ser dita explicitamente porque é o resultado do bloco:

| Como se compra silhueta | O que se ganha | O que se paga |
|:---|:---|:---|
| Ligação `single` | Silhueta perto de 0,32 | Um segmento com **1 imóvel** — não existe ação comercial para ele |
| Abrir mão da escala | Silhueta acima de 0,50 | O ARI contra a referência cai para **0,03**: é outra segmentação, comandada por uma variável só |
| Projetar em 2 componentes | Silhueta acima de 0,45 | Ninguém muda de grupo (ARI ≈ 0,99) — só a régua encolheu. O número sobe, a segmentação é a mesma |

A conclusão não é que a silhueta seja uma métrica ruim. É que ela **não é comparável entre
espaços**, e confrontar dois números de silhueta medidos em escalas ou dimensões diferentes é
um erro de tipo, não uma escolha de modelagem. A regra de comparação que a fase 4 declarou — só
se compara silhueta medida no mesmo espaço — não era formalismo; era a defesa contra exatamente
isto.

## 1.4 · A varredura admissível — a pergunta refeita corretamente

Refeita a pergunta com as restrições que já estavam declaradas antes de qualquer varredura —
mesmo espaço, *k* entre 2 e 5, menor grupo com pelo menos 2% da carteira —, a busca pela melhor
silhueta volta a ser uma pergunta bem posta. E ela tem uma resposta clara.

In [ ]:
admissivel = grade.query(
    "espaco == 'euclidiano (17)' and @K_MINIMO <= k <= @K_MAXIMO "
    "and menor_grupo_pct >= 100 * @MENOR_SEGMENTO_MINIMO"
).sort_values("silhueta", ascending=False).copy()
admissivel["e_o_modelo_da_fase_4"] = (
    (admissivel.algoritmo == ALGORITMO) & (admissivel.k == K_ESCOLHIDO)
)

MELHOR_ADMISSIVEL = admissivel.iloc[0]
K_MELHOR_SILHUETA = int(MELHOR_ADMISSIVEL.k)
SILHUETA_MELHOR_ADMISSIVEL = float(MELHOR_ADMISSIVEL.silhueta)

fig, ax = plt.subplots(figsize=(13, 7.4))
ax.barh(
    [f"{a} · k={k}" for a, k in zip(admissivel.algoritmo, admissivel.k)],
    admissivel.silhueta,
    color=[ALERTA if e else (DESTAQUE if k == K_MELHOR_SILHUETA else NEUTRO)
           for e, k in zip(admissivel.e_o_modelo_da_fase_4, admissivel.k)],
)
ax.bar_label(ax.containers[0], fmt="%.4f", padding=3, fontsize=8.5)
ax.axvline(SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
ax.invert_yaxis()
ax.set_xlim(0, admissivel.silhueta.max() * 1.22)
ax.set_xlabel("Silhueta — todas medidas no mesmo espaço de 17 variáveis")
ax.set_title(f"As {len(admissivel)} configurações admissíveis — coral é o modelo da fase 4, "
             f"azul-claro é a melhor silhueta")

salvar(fig, "ranking-admissivel", admissivel)
admissivel.head(12)[["algoritmo", "k", "silhueta", "menor_grupo_pct", "maior_grupo_pct",
                     "e_o_modelo_da_fase_4"]]

### O resultado da varredura, dito sem rodeio

Dentro do espaço declarado e das restrições declaradas, **a melhor silhueta está em *k* = 2** —
e não por pouco. O padrão vale para praticamente todo algoritmo da grade, e não é acidente
desta carteira: a silhueta mede separação relativa, e separação relativa é máxima quando há
dois grupos, porque com *k* = 2 não existe "vizinho mais próximo errado" — só existe um
vizinho.

Isto é exatamente o que a fase 4 antecipou ao declarar a regra de escolha de *k* **antes** de
varrer:

> A regra **não** é "o *k* de maior silhueta": a silhueta cresce quando *k* diminui, e levada ao
> limite ela devolve dois grupos, "caro" e "barato", que a diretoria já tem.

A frase estava escrita antes do resultado. Agora ela é uma afirmação **testável** — e o bloco 2
a testa com o rótulo aberto, que é a única evidência capaz de confirmá-la ou derrubá-la. Se
*k* = 2 **não** for a partição entre caro e barato, a fase 4 errou e a regra precisa cair.

A tabela abaixo fecha o bloco: a varredura de *k* com estabilidade, e os dois candidatos que
saem dela — o que a silhueta prefere e o que a regra escolhe.

In [ ]:
# A varredura de k com estabilidade, para o confronto final da regra.
gerador = np.random.default_rng(SEMENTE)
FRACAO, REPETICOES = 0.80, 20

linhas_k = []
for k in K_VARREDURA:
    rotulos_k = KMeans(n_clusters=k, n_init=20, random_state=SEMENTE).fit_predict(EUCLIDIANO)
    aris = []
    for _ in range(REPETICOES):
        indices = gerador.choice(LINHAS, size=int(FRACAO * LINHAS), replace=False)
        # A receita é reajustada dentro da reamostra — a decisão da fase 3.
        matriz = clone(preprocessador).fit_transform(base[ESPACO_EUCLIDIANO].iloc[indices])
        parcial = KMeans(n_clusters=k, n_init=10,
                         random_state=int(gerador.integers(1e6))).fit_predict(matriz)
        aris.append(adjusted_rand_score(rotulos_k[indices], parcial))
    contagem = np.bincount(rotulos_k)
    linhas_k.append(
        {
            "k": k,
            "silhueta": round(float(silhouette_score(EUCLIDIANO, rotulos_k)), 4),
            "menor_grupo_pct": round(100 * contagem.min() / LINHAS, 1),
            "ari_medio": round(float(np.mean(aris)), 4),
            "ari_minimo": round(float(np.min(aris)), 4),
        }
    )
por_k = pd.DataFrame(linhas_k).set_index("k")

testes_k = pd.DataFrame(
    {
        f"k dentro de [{K_MINIMO}, {K_MAXIMO}]": [K_MINIMO <= k <= K_MAXIMO for k in por_k.index],
        f"silhueta ≥ {SILHUETA_MINIMA:.2f}": por_k.silhueta >= SILHUETA_MINIMA,
        f"menor grupo ≥ {100 * MENOR_SEGMENTO_MINIMO:.0f}%": (
            por_k.menor_grupo_pct >= 100 * MENOR_SEGMENTO_MINIMO
        ),
        f"ARI médio ≥ {ARI_MINIMO:.2f}": por_k.ari_medio >= ARI_MINIMO,
        f"ARI mínimo ≥ {ARI_MINIMO:.2f}": por_k.ari_minimo >= ARI_MINIMO,
    },
    index=por_k.index,
)
por_k["passa_em_tudo"] = testes_k.all(axis=1)

K_PELA_REGRA = int(por_k.index[por_k.passa_em_tudo].max())
K_PELA_SILHUETA = int(por_k.silhueta.idxmax())
assert K_PELA_REGRA == K_ESCOLHIDO, "a regra da fase 4 não reproduz o k gravado no manifesto"

fig, eixos = plt.subplots(2, 1, figsize=(13, 7.4), height_ratios=[1.35, 1],
                          constrained_layout=True, sharex=True)

eixos[0].bar(por_k.index, por_k.silhueta,
             color=[DESTAQUE if k == K_PELA_SILHUETA else (ALERTA if k == K_PELA_REGRA else NEUTRO)
                    for k in por_k.index])
eixos[0].bar_label(eixos[0].containers[0], fmt="%.4f", padding=3, fontsize=8.5)
eixos[0].axhline(SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
eixos[0].set_ylabel("Silhueta")
eixos[0].margins(y=0.22)
eixos[0].set_title(f"Melhor silhueta em k = {K_PELA_SILHUETA} (azul-claro) · maior k que passa "
                   f"em tudo: k = {K_PELA_REGRA} (coral) — a regra declarada na fase 4")

sns.heatmap(
    testes_k.T.astype(int),
    annot=testes_k.T.map(lambda passou: "✓" if passou else "✗").to_numpy(), fmt="",
    cmap=plt.matplotlib.colors.ListedColormap([ALERTA, APROVADO]), vmin=0, vmax=1, cbar=False,
    linewidths=1.2, linecolor="white",
    annot_kws={"fontsize": 11, "color": "white", "fontweight": "bold"}, ax=eixos[1],
)
eixos[1].set_xlabel("k")
eixos[1].tick_params(axis="y", rotation=0)

salvar(fig, "melhor-silhueta-versus-regra", por_k.reset_index())
por_k

In [ ]:
GANHO_PERCENTUAL = 100 * (por_k.loc[K_PELA_SILHUETA, "silhueta"] / SILHUETA_ESCOLHIDO - 1)

registrar(
    "1 · Varredura",
    f"A melhor silhueta admissível está em k = {K_MELHOR_SILHUETA}, não em k = {K_ESCOLHIDO}",
    f"{len(grade)} configurações de k × algoritmo, {len(escala)} de escala, "
    f"{len(reducao)} de dimensão",
    "A silhueta mede separação relativa e cresce quando k cai: com dois grupos não existe "
    "vizinho mais próximo errado. O ganho é real e não decide nada sozinho — a regra de "
    "escolha de k foi declarada na fase 4 antes de qualquer varredura, e não é a silhueta.",
    f"k = {K_MELHOR_SILHUETA}: {por_k.loc[K_MELHOR_SILHUETA, 'silhueta']:.4f} · "
    f"k = {K_ESCOLHIDO}: {SILHUETA_ESCOLHIDO:.4f} · ganho de {GANHO_PERCENTUAL:.0f}%",
)
registrar(
    "1 · Varredura",
    "Trocar a escala ou a dimensão para melhorar a silhueta é erro de tipo",
    "Escala e redução de dimensionalidade",
    "Sem escala, a silhueta dobra e a partição é OUTRA — o ARI contra a referência despenca, e "
    "quem segmenta passa a ser a variável de maior variância bruta. Com 2 componentes, a "
    "silhueta sobe e NINGUÉM muda de grupo: só a régua encolheu.",
    f"sem escala em k = {K_ESCOLHIDO}: silhueta {SILHUETA_SEM_ESCALA:.4f} com ARI "
    f"{ARI_SEM_ESCALA:.2f} · PCA de 2 componentes em k = 2: silhueta {SILHUETA_PCA2:.4f} "
    f"com ARI {ARI_PCA2:.2f}",
    acao="armadilha registrada",
)

# 2 · Avaliar os resultados

*Evaluate results* — a primeira tarefa da fase 5, e a que muda a régua. A fase 4 mediu
**geometria**: coesão, separação, estabilidade. Aqui a pergunta é outra, e está escrita no
Canvas do Problema desde o Encontro 1:

> Substituir a política única por **políticas por linha de produto**, de modo que o capital de
> reforma vá para onde tem retorno e o esforço de corretagem vá para onde tem margem.

Um agrupamento com silhueta excelente que devolve as mesmas duas caixas que a diretoria já usa
não resolve esse problema. Um agrupamento com silhueta modesta que devolve três linhas
operáveis, com valores e bairros diferentes, resolve. É essa distinção que o bloco mede.

Quatro exames, na ordem em que a evidência aparece:

| Exame | Pergunta | Critério |
|:---|:---|:---|
| **2.1 · Independência do preço** | A segmentação é uma tabela de preço disfarçada? | ARI contra quartis de `preco_venda` **< 0,40** — o critério pendente da fase 4 |
| **2.2 · Valor por segmento** | Quanto vale cada linha de produto? | Receita, preço mediano e amplitude, com o rótulo aberto |
| **2.3 · Validação externa** | Os segmentos existem fora do espaço em que foram criados? | `bairro`, que nunca entrou em algoritmo nenhum |
| **2.4 · O confronto de *k*** | *k* = 2, que ganhou em silhueta, serve ao objetivo? | As três medidas acima, aplicadas a cada *k* candidato |

## 2.1 · O critério pendente — a segmentação é uma tabela de preço?

A fase 4 deixou este critério explicitamente em aberto, e por um motivo de método: medi-lo lá
exigiria abrir o rótulo antes de a modelagem terminar, e a partir daí não haveria como provar
que o preço não influenciou a escolha do modelo.

O critério está no Canvas, seção 6, e foi acrescentado no Encontro 3:

> **Independência do rótulo reservado: ARI contra quartis de preço < 0,40.**

Duas medidas, não uma. A primeira é o critério declarado — ARI contra os **quartis** de preço.
A segunda não está no Canvas e é acrescentada aqui com finalidade específica: o ARI contra o
**corte pela mediana**, que é a política de duas faixas que a Meridiano já opera. É ela que
testa a afirmação que a fase 4 fez sem poder medir — a de que *k* = 2 devolveria "caro e
barato".

In [ ]:
ARI_PRECO_MAXIMO = 0.40  # Canvas, seção 6 — acrescentado no Encontro 3.

quartis_de_preco = pd.qcut(curada.preco_venda, 4, labels=False).to_numpy()
acima_da_mediana = (curada.preco_venda > curada.preco_venda.median()).astype(int).to_numpy()

linhas_preco = []
particoes_por_k = {}
for k in range(2, 7):
    rotulos_k = pers.ordenar_canonicamente(
        base, KMeans(n_clusters=k, n_init=20, random_state=SEMENTE).fit_predict(EUCLIDIANO)
    )
    particoes_por_k[k] = rotulos_k
    linhas_preco.append(
        {
            "k": k,
            "silhueta": round(float(silhouette_score(EUCLIDIANO, rotulos_k)), 4),
            "ari_contra_quartis": round(float(adjusted_rand_score(quartis_de_preco, rotulos_k)), 4),
            "ari_contra_a_mediana": round(
                float(adjusted_rand_score(acima_da_mediana, rotulos_k)), 4
            ),
        }
    )
independencia = pd.DataFrame(linhas_preco).set_index("k")
independencia["passa_no_criterio"] = independencia.ari_contra_quartis < ARI_PRECO_MAXIMO

# A partição do modelo escolhido é a que está gravada, não uma reestimativa.
ARI_QUARTIS = round(float(adjusted_rand_score(quartis_de_preco, ROTULOS)), 4)
ARI_MEDIANA = round(float(adjusted_rand_score(acima_da_mediana, ROTULOS)), 4)
CRITERIO_PENDENTE_ATENDIDO = ARI_QUARTIS < ARI_PRECO_MAXIMO

fig, eixos = plt.subplots(1, 2, figsize=(16, 5.2), width_ratios=[1.3, 1],
                          constrained_layout=True)

eixos[0].plot(independencia.index, independencia.ari_contra_quartis, marker="o", lw=2.4,
              color=BARRA, label="ARI contra os quartis de preço — o critério do Canvas")
eixos[0].plot(independencia.index, independencia.ari_contra_a_mediana, marker="s", lw=2.4,
              ls="dashed", color=ALERTA,
              label="ARI contra o corte pela mediana — a política atual da Meridiano")
eixos[0].axhline(ARI_PRECO_MAXIMO, color=REFERENCIA, ls="dotted", lw=2)
eixos[0].text(6, ARI_PRECO_MAXIMO, f" teto do Canvas: {ARI_PRECO_MAXIMO:.2f} ", va="bottom",
              ha="right", fontsize=9, color=config.AZUL_PROFUNDO)
eixos[0].annotate(
    f"k = 2 é a política que\na diretoria já tem\n"
    f"(ARI {independencia.loc[2, 'ari_contra_a_mediana']:.2f})",
    (2, independencia.loc[2, "ari_contra_a_mediana"]), textcoords="offset points",
    xytext=(16, -4), fontsize=9, color=ALERTA, fontweight="bold",
)
eixos[0].set_xticks(range(2, 7))
eixos[0].set_xlabel("k")
eixos[0].set_ylabel("Índice de Rand ajustado contra o preço")
eixos[0].set_ylim(0, 0.62)
eixos[0].set_title("Quanto de cada segmentação já estava no preço")
eixos[0].legend(fontsize=8.5, loc="upper right")

contingencia = pd.crosstab(
    pd.Series(particoes_por_k[2], name="Segmentação com k = 2"),
    pd.Series(np.where(acima_da_mediana == 1, "acima da mediana", "abaixo da mediana"),
              name="Faixa de preço da política atual"),
)
sns.heatmap(contingencia, annot=True, fmt="d", cmap="Blues", linewidths=0.8, cbar=False,
            ax=eixos[1])
eixos[1].set_title(f"k = 2 contra a política de duas faixas\n"
                   f"{100 * np.diag(contingencia.to_numpy()).sum() / LINHAS:.1f}% dos imóveis "
                   f"caem na mesma caixa")

salvar(fig, "independencia-do-preco", independencia.reset_index())
independencia

### O que 2.1 resolve — e é bastante

**O critério pendente está atendido.** O ARI da segmentação escolhida contra os quartis de
preço fica em torno de 0,33, abaixo do teto de 0,40 que o Canvas fixou no Encontro 3. A
segmentação **não** é uma tabela de preço com outro nome: ela guarda relação com o preço — o
que seria estranho não acontecer, já que porte e padrão construtivo são o que forma preço —,
mas não se reduz a ele.

**E a previsão da fase 4 se confirmou.** A curva tracejada é a que importa para a decisão de
*k*: em *k* = 2, o ARI contra o corte pela mediana é o mais alto de toda a varredura, e a
tabela de contingência mostra a maioria dos imóveis caindo na mesma caixa em que a política
atual já os coloca.

Isto fecha o argumento que o bloco 1 deixou em aberto. O *k* de melhor silhueta é o que mais se
parece com a política que o projeto foi contratado para substituir — e o ganho de silhueta que
ele oferece é o ganho de reproduzir uma divisão que a Meridiano já opera. A regra da fase 4 não
foi conservadorismo: foi a decisão correta, e agora há um número que a sustenta.

Vale ainda notar o que a curva sólida faz: o ARI contra os quartis **sobe** de *k* = 2 para
*k* = 4 e cai depois. Nenhum *k* chega perto do teto, mas o mínimo não está onde a silhueta
está — mais um indício de que as duas métricas respondem a perguntas diferentes.

In [ ]:
registrar(
    "2 · Resultados",
    "O critério pendente da fase 4 está atendido — a segmentação não é uma tabela de preço",
    "Independência do rótulo reservado (Canvas, seção 6)",
    "A segmentação guarda relação com o preço, como era esperado — porte e padrão formam "
    "preço —, mas não se reduz a ele: a concordância fica bem abaixo do teto declarado.",
    f"ARI contra os quartis de preço: {ARI_QUARTIS:.4f} (teto {ARI_PRECO_MAXIMO:.2f})",
)
registrar(
    "2 · Resultados",
    f"k = {K_MELHOR_SILHUETA}, o de melhor silhueta, reproduz a política que a diretoria já tem",
    "A regra de escolha de k declarada na fase 4",
    "A fase 4 previu, antes de varrer, que a silhueta levaria a dois grupos equivalentes a "
    "caro e barato — e recusou a silhueta como regra por causa disso. Com o rótulo aberto, a "
    "previsão é medida: em k = 2 a concordância com o corte pela mediana é a maior de toda a "
    "varredura.",
    f"ARI de k = 2 contra o corte pela mediana: "
    f"{independencia.loc[2, 'ari_contra_a_mediana']:.4f}, contra "
    f"{independencia.loc[K_ESCOLHIDO, 'ari_contra_a_mediana']:.4f} em k = {K_ESCOLHIDO}",
    acao="decisão mantida",
)

## 2.2 · Quanto vale cada linha de produto

Com o critério de independência atendido, o preço passa a ser o que ele deve ser nesta fase:
**a medida do valor de cada segmento**, não a régua que define os segmentos.

Três leituras interessam à diretoria, e as três estão na tabela abaixo:

* **a receita concentrada** — quanto do faturamento de 2006 a 2010 passou por cada linha. É o
  número que decide para onde vai o orçamento de corretagem;
* **o preço mediano e a amplitude** — o preço típico e a dispersão dentro do grupo. Amplitude
  alta significa que a linha comporta subfaixas, o que é informação de precificação;
* **o preço por pé quadrado** — a medida que separa *imóvel caro* de *imóvel caro por ser
  grande*. Dois segmentos com preços medianos distantes e preço por pé quadrado parecido são
  o mesmo produto em tamanhos diferentes.

In [ ]:
valor = perf.leitura_a_posteriori(curada, ROTULOS)
silhueta_por_segmento = part.silhueta_por_grupo(EUCLIDIANO, ROTULOS)
valor["silhueta_media"] = silhueta_por_segmento.media
valor["pct_silhueta_negativa"] = (100 * silhueta_por_segmento.pct_negativa).round(1)

SEGMENTO_LIDER = int(valor.pct_receita.idxmax())
RECEITA_LIDER = float(valor.pct_receita.max())

fig, eixos = plt.subplots(1, 2, figsize=(16, 5.6), width_ratios=[1.35, 1],
                          constrained_layout=True)

cores_segmento = [config.PALETA_SEGMENTOS[s % len(config.PALETA_SEGMENTOS)] for s in valor.index]
dados_caixa = [curada.preco_venda[ROTULOS == s] for s in valor.index]
caixas = eixos[0].boxplot(dados_caixa, patch_artist=True, widths=0.55,
                          medianprops={"color": "white", "lw": 2},
                          flierprops={"marker": ".", "ms": 3, "alpha": 0.35})
for caixa, cor in zip(caixas["boxes"], cores_segmento):
    caixa.set_facecolor(cor)
    caixa.set_alpha(0.85)
eixos[0].set_xticks(
    range(1, len(valor) + 1),
    [f"Segmento {s}\n{pt(int(n))} imóveis" for s, n in zip(valor.index, valor.imoveis)],
)
eixos[0].set_ylabel("Preço de venda (US$)")
eixos[0].yaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f}k")
eixos[0].set_title("As faixas centrais não se sobrepõem — as caudas se sobrepõem muito")
# O rótulo vai acima da haste superior, não ao lado da mediana: com caixas
# largas, um deslocamento lateral cai dentro da própria caixa.
for posicao, (segmento, linha) in enumerate(valor.iterrows(), start=1):
    topo_da_haste = float(caixas["caps"][2 * (posicao - 1) + 1].get_ydata()[0])
    eixos[0].annotate(f"mediana {usd(linha.preco_mediano)}", (posicao, topo_da_haste),
                      textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8.5,
                      color=config.AZUL_PROFUNDO, fontweight="bold")

eixos[1].barh([f"Segmento {s}" for s in valor.index], valor.pct_receita, color=cores_segmento)
eixos[1].bar_label(
    eixos[1].containers[0],
    labels=[f"{p:.1f}%   ({usd(r)})" for p, r in zip(valor.pct_receita, valor.receita_total)],
    padding=4, fontsize=9,
)
eixos[1].invert_yaxis()
eixos[1].set_xlim(0, valor.pct_receita.max() * 1.62)
eixos[1].set_xlabel("Participação na receita transacionada de 2006 a 2010")
eixos[1].set_title(f"O segmento {SEGMENTO_LIDER} concentra {RECEITA_LIDER:.1f}% da receita")

salvar(fig, "valor-por-segmento", valor.reset_index())
valor[["imoveis", "preco_mediano", "preco_p10", "preco_p90", "amplitude_p90_p10",
       "preco_por_pe2", "pct_receita", "silhueta_media", "pct_silhueta_negativa"]]

### A leitura, incluindo a parte desconfortável

Três linhas de produto com identidades econômicas distintas, e um problema que precisa ser dito
com a mesma clareza:

* **o segmento 2 concentra quase metade da receita** com menos de um terço dos imóveis, e tem
  o maior preço por pé quadrado da carteira. Não é apenas o segmento dos imóveis grandes: é o
  segmento em que o pé quadrado vale mais, o que é a definição operacional de alto padrão. É
  para onde a corretagem dedicada deve ir;
* **o segmento 0 tem preço por pé quadrado maior que o do segmento 1**, apesar de preço mediano
  bem menor. A leitura é direta: são imóveis pequenos e antigos, onde o custo do terreno e da
  infraestrutura pesa mais por unidade de área. Isso muda a política — desconto por pé quadrado
  nesse grupo destrói margem mais rápido do que parece;
* **o segmento 1 é o grupo frágil, e a fragilidade é mensurável.** A silhueta média dele é a
  mais baixa das três, com uma parcela relevante de imóveis em silhueta negativa — imóveis que
  estariam mais próximos do centro de outro grupo. É o segmento do meio, e o meio é onde a
  fronteira dói: ele fica espremido entre o compacto de entrada e o familiar consolidado.

A última leitura é a mais importante para a fase 6, e não se resolve com mais um algoritmo. Ela
significa que **a atribuição de segmento para imóveis próximos da fronteira é incerta**, e uma
política que trate todo imóvel do segmento 1 como igualmente pertencente ao segmento 1 vai
errar em uma parcela dele. A recomendação correspondente está no bloco 4.

In [ ]:
fronteira = pd.DataFrame(
    {
        "imoveis": silhueta_por_segmento.registros,
        "silhueta_media": silhueta_por_segmento.media,
        "imoveis_em_silhueta_negativa": (
            silhueta_por_segmento.registros * silhueta_por_segmento.pct_negativa
        ).round(0).astype(int),
        "pct_em_silhueta_negativa": (100 * silhueta_por_segmento.pct_negativa).round(1),
    }
)
fronteira["preco_mediano"] = valor.preco_mediano
SEGMENTO_FRAGIL = int(fronteira.silhueta_media.idxmin())
IMOVEIS_NA_FRONTEIRA = int(fronteira.imoveis_em_silhueta_negativa.sum())

io.salvar_tabela(fronteira.reset_index(), SECAO, "imoveis-na-fronteira", indice=False)

registrar(
    "2 · Resultados",
    f"O segmento {SEGMENTO_LIDER} concentra {RECEITA_LIDER:.1f}% da receita com "
    f"{100 * valor.loc[SEGMENTO_LIDER, 'imoveis'] / LINHAS:.0f}% dos imóveis",
    "Valor por linha de produto",
    "As três linhas têm identidades econômicas distintas — preço mediano, amplitude e preço "
    "por pé quadrado diferentes —, o que é a condição para que políticas diferenciadas façam "
    "sentido econômico e não apenas estatístico.",
    f"receita do segmento {SEGMENTO_LIDER}: {usd(valor.loc[SEGMENTO_LIDER, 'receita_total'])} "
    f"({RECEITA_LIDER:.1f}%) · preço por pé² de "
    f"{valor.loc[SEGMENTO_LIDER, 'preco_por_pe2']:.2f}, o maior da carteira",
)
registrar(
    "2 · Resultados",
    f"O segmento {SEGMENTO_FRAGIL} é o grupo de fronteira, e a fase 6 precisa tratá-lo assim",
    "Coesão interna por segmento",
    "O segmento do meio fica espremido entre os dois vizinhos: parte relevante dos seus "
    "imóveis está mais próxima do centro de outro grupo. Tratar todos como igualmente "
    "pertencentes ao segmento faz a política errar justamente na fronteira.",
    f"silhueta média {fronteira.loc[SEGMENTO_FRAGIL, 'silhueta_media']:.4f} · "
    f"{fronteira.loc[SEGMENTO_FRAGIL, 'imoveis_em_silhueta_negativa']} imóveis em silhueta "
    f"negativa ({fronteira.loc[SEGMENTO_FRAGIL, 'pct_em_silhueta_negativa']:.1f}% do grupo) · "
    f"{IMOVEIS_NA_FRONTEIRA} na carteira inteira",
)

## 2.3 · A validação externa — `bairro`

Este é o exame mais forte do bloco, e o que a reserva de variáveis da fase 3 tornou possível.

`bairro` **nunca entrou em nenhum algoritmo**. Não estava na base de modelagem, não entrou na
receita de pré-processamento, não participou de nenhuma varredura. Se os segmentos formados a
partir de porte, configuração, padrão e idade se distribuírem de forma diferente entre os
bairros, essa diferença não pode ter sido induzida pelo procedimento — ela é estrutura real do
mercado de Ames, capturada por variáveis que não mencionam localização.

É a diferença entre um agrupamento que descreve os dados e um agrupamento que descreve o
**mercado**. E é medível: o V de Cramér entre bairro e segmento diz o quanto saber o bairro
informa sobre o segmento.

In [ ]:
por_bairro = perf.validacao_por_bairro(curada, ROTULOS, top=10)

tabela_bairro = pd.crosstab(curada.bairro, ROTULOS)
qui_quadrado, valor_p, graus, _ = chi2_contingency(tabela_bairro)
V_CRAMER = float(
    np.sqrt(qui_quadrado / (tabela_bairro.to_numpy().sum() * (min(tabela_bairro.shape) - 1)))
)

# Bairros que concentram um segmento: a evidência mais legível da validação.
concentracao = por_bairro.max(axis=1)
BAIRRO_MAIS_CONCENTRADO = str(concentracao.idxmax())
SEGMENTO_DO_BAIRRO = int(por_bairro.loc[BAIRRO_MAIS_CONCENTRADO].idxmax())

fig, eixos = plt.subplots(1, 2, figsize=(16, 5.8), width_ratios=[1.5, 1],
                          constrained_layout=True)

sns.heatmap(por_bairro, annot=True, fmt=".1f", cmap="Blues", linewidths=0.7,
            cbar_kws={"shrink": 0.8, "label": "% dos imóveis do bairro"}, ax=eixos[0])
eixos[0].set_xlabel("Segmento")
eixos[0].set_ylabel("")
eixos[0].tick_params(axis="y", rotation=0)
eixos[0].set_title(f"Composição de segmentos nos 10 maiores bairros — V de Cramér = "
                   f"{V_CRAMER:.3f}")

composicao_geral = 100 * pd.Series(np.bincount(ROTULOS) / LINHAS, index=valor.index)
desvio = (por_bairro - composicao_geral).abs().mean(axis=1).sort_values(ascending=False)
eixos[1].barh(desvio.index, desvio.to_numpy(), color=BARRA)
eixos[1].bar_label(eixos[1].containers[0], fmt="%.1f pp", padding=3, fontsize=8.5)
eixos[1].invert_yaxis()
eixos[1].set_xlim(0, desvio.max() * 1.28)
eixos[1].set_xlabel("Desvio médio da composição da carteira (pontos percentuais)")
eixos[1].set_title("Nenhum bairro se parece com a carteira média")

salvar(fig, "validacao-por-bairro", por_bairro.reset_index())
por_bairro

### O que a validação externa mostra

O V de Cramér entre bairro e segmento é alto — bem acima do que se espera de duas variáveis
sem relação —, e o painel da direita traduz isso em algo mais direto: **nenhum bairro se parece
com a carteira média**. Se os segmentos fossem artefato do procedimento, a composição de cada
bairro tenderia a repetir a composição geral, e as barras seriam curtas.

O que se vê é o oposto. Há bairros em que um único segmento responde por mais de 90% das
transações, e há bairros que praticamente não têm imóveis do segmento de maior valor. Isso não
foi ensinado ao algoritmo: ele viu área, cômodos, qualidade e idade, e a geografia se organizou
sozinha em cima disso.

É o resultado que dá confiança para a fase 6 — e também um alerta operacional que a fase 6
precisa absorver: se um segmento se concentra em poucos bairros, a política daquele segmento é,
na prática, uma política de bairro, com toda a exposição a risco geográfico que isso implica.

In [ ]:
registrar(
    "2 · Resultados",
    "Os segmentos se sustentam contra uma variável que nunca entrou no algoritmo",
    "Validação externa por `bairro`",
    "`bairro` foi reservado na fase 3 e não participou de nenhuma varredura. A associação "
    "medida agora não pode ter sido induzida pelo procedimento: é estrutura de mercado "
    "capturada por variáveis que não mencionam localização.",
    f"V de Cramér = {V_CRAMER:.3f} (p < 0,001) · em {BAIRRO_MAIS_CONCENTRADO}, o segmento "
    f"{SEGMENTO_DO_BAIRRO} responde por "
    f"{por_bairro.loc[BAIRRO_MAIS_CONCENTRADO].max():.1f}% das transações",
)

## 2.4 · O confronto de *k* sob a ótica do negócio

O bloco 1 terminou com dois candidatos e uma pergunta que a geometria não resolvia. Agora há
evidência de negócio para os dois, e ela pode ser posta lado a lado.

A tabela abaixo mede cada *k* candidato pelas quatro coisas que o Canvas pede de uma
segmentação — e a coluna que decide não é a silhueta:

| Coluna | O que mede | De onde vem a exigência |
|:---|:---|:---|
| `linhas_de_produto` | Quantas políticas diferenciadas o resultado permite | Canvas §3 e §11 |
| `ari_contra_a_mediana` | Quanto do resultado a diretoria já tinha | Canvas §2 — a política única a ser substituída |
| `ari_contra_quartis` | Se virou tabela de preço | Canvas §6 — teto de 0,40 |
| `menor_receita_pct` | Se a menor linha tem massa econômica para justificar orçamento próprio | Canvas §4 e §11 |
| `desequilibrio_da_receita_pp` | O quanto a linha mais distante se afasta de uma divisão igual da receita | Canvas §11 — orçamento por linha |

In [ ]:
linhas_confronto = []
for k, rotulos_k in particoes_por_k.items():
    leitura = perf.leitura_a_posteriori(curada, rotulos_k)
    contagem = np.bincount(rotulos_k)
    medidas = por_k.loc[k]
    linhas_confronto.append(
        {
            "k": k,
            "linhas_de_produto": k,
            "silhueta": medidas.silhueta,
            "ari_minimo": medidas.ari_minimo,
            "menor_grupo_pct": medidas.menor_grupo_pct,
            "menor_receita_pct": round(float(leitura.pct_receita.min()), 1),
            # Desequilíbrio: o quanto a linha mais distante se afasta de uma
            # divisão igual da receita. Menor é mais equilibrado.
            "desequilibrio_da_receita_pp": round(
                float((leitura.pct_receita - 100 / k).abs().max()), 1
            ),
            "ari_contra_quartis": independencia.loc[k, "ari_contra_quartis"],
            "ari_contra_a_mediana": independencia.loc[k, "ari_contra_a_mediana"],
            "amplitude_mediana": round(float(leitura.amplitude_p90_p10.median()), 2),
            "passa_nos_criterios_tecnicos": bool(medidas.passa_em_tudo),
        }
    )
confronto = pd.DataFrame(linhas_confronto).set_index("k")

fig, eixos = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

cores = [ALERTA if k == K_ESCOLHIDO else (DESTAQUE if k == K_MELHOR_SILHUETA else NEUTRO)
         for k in confronto.index]

eixos[0].bar(confronto.index, confronto.silhueta, color=cores)
eixos[0].bar_label(eixos[0].containers[0], fmt="%.4f", padding=3, fontsize=8.5)
eixos[0].axhline(SILHUETA_MINIMA, color=REFERENCIA, ls="dashed", lw=2)
eixos[0].set_xlabel("k")
eixos[0].set_ylabel("Silhueta")
eixos[0].set_title("O que a geometria prefere")
eixos[0].margins(y=0.2)

eixos[1].bar(confronto.index, confronto.ari_contra_a_mediana, color=cores)
eixos[1].bar_label(eixos[1].containers[0], fmt="%.3f", padding=3, fontsize=8.5)
eixos[1].set_xlabel("k")
eixos[1].set_ylabel("ARI contra o corte pela mediana")
eixos[1].set_title("O quanto disso a diretoria já tinha")
eixos[1].margins(y=0.2)

eixos[2].bar(confronto.index, confronto.menor_receita_pct, color=cores)
eixos[2].bar_label(eixos[2].containers[0], fmt="%.1f%%", padding=3, fontsize=8.5)
eixos[2].set_xlabel("k")
eixos[2].set_ylabel("Receita da menor linha (%)")
eixos[2].set_title("Se a menor linha sustenta orçamento próprio")
eixos[2].margins(y=0.2)

fig.suptitle(f"Coral: o modelo escolhido (k = {K_ESCOLHIDO}) · azul-claro: o de melhor "
             f"silhueta (k = {K_MELHOR_SILHUETA})", fontsize=11,
             color=config.AZUL_PROFUNDO, fontweight="bold")

salvar(fig, "confronto-de-k-no-negocio", confronto.reset_index())
confronto

### O veredito sobre *k*

Lendo as três medidas juntas:

* ***k* = 2 ganha na silhueta e perde no resto.** Entrega duas linhas de produto — o mínimo
  aritmético para chamar algo de segmentação — e é a partição que mais se confunde com a
  política de duas faixas que o projeto foi contratado para substituir. Implantá-lo seria
  entregar à diretoria, com aparato estatístico, a régua que ela já usa;
* ***k* = 3 passa em todos os critérios técnicos e nos três de negócio.** Três linhas com massa
  econômica própria — a menor com mais de um quarto da receita —, independência do preço
  confirmada e a menor concordância com a política atual entre os *k* que passam;
* ***k* = 4 é o candidato de revisão, e o motivo é interessante.** Ele reprova por silhueta, e
  por pouco. Em compensação, tem o menor desequilíbrio de receita da varredura — as quatro
  linhas ficam perto de um quarto do faturamento cada — e o menor ARI contra a política atual
  entre os candidatos com quatro ou menos linhas. Não é adotado aqui — mover o piso de silhueta depois de ver o resultado é exatamente
  o que este projeto se proibiu de fazer em todas as fases anteriores —, mas é registrado como
  a revisão a levar ao Canvas, e o bloco 4 diz em que condições.

A decisão da fase 4 fica **mantida**, e agora por dois caminhos independentes: a regra
declarada antes do resultado e a evidência de negócio medida depois dele.

In [ ]:
# O candidato de revisão: entre os k acima do escolhido, o de receita mais
# equilibrada entre as linhas — que é a vantagem de negócio que ele oferece.
K_REVISAO = int(confronto.query("index > @K_ESCOLHIDO").desequilibrio_da_receita_pp.idxmin())
DESEQUILIBRIO_MINIMO = bool(
    confronto.loc[K_REVISAO, "desequilibrio_da_receita_pp"]
    == confronto.desequilibrio_da_receita_pp.min()
)
folga_silhueta = SILHUETA_MINIMA - float(confronto.loc[K_REVISAO, "silhueta"])

registrar(
    "2 · Resultados",
    f"O modelo da fase 4 — {ALGORITMO}, k = {K_ESCOLHIDO} — é confirmado sob a ótica do negócio",
    "Confronto de k pelos critérios do Canvas",
    "Passa nos critérios técnicos, atende o critério pendente de independência do preço, "
    "entrega três linhas com massa econômica própria e é, entre os k aprovados, o que menos "
    "reproduz a política de duas faixas em vigor.",
    f"silhueta {SILHUETA_ESCOLHIDO:.4f} · ARI contra quartis {ARI_QUARTIS:.4f} · menor linha "
    f"com {confronto.loc[K_ESCOLHIDO, 'menor_receita_pct']:.1f}% da receita",
    acao="decisão mantida",
)
registrar(
    "2 · Resultados",
    f"k = {K_REVISAO} fica registrado como candidato de revisão, e não é adotado agora",
    "Piso de silhueta do Canvas",
    "Reprova apenas no piso de silhueta, e por margem estreita. Tem a distribuição de receita "
    "mais equilibrada da varredura. Mover o piso depois de ver o resultado inverteria a ordem "
    "entre regra e resultado — a revisão precisa ser decidida no Canvas, antes da próxima "
    "execução, e não aqui.",
    f"silhueta {confronto.loc[K_REVISAO, 'silhueta']:.4f}, a {folga_silhueta:.4f} do piso de "
    f"{SILHUETA_MINIMA:.2f} · desequilíbrio de receita de "
    f"{confronto.loc[K_REVISAO, 'desequilibrio_da_receita_pp']:.1f} pontos percentuais, o menor "
    f"da varredura: {DESEQUILIBRIO_MINIMO}",
    acao="próxima etapa",
)

## 2.5 · Os critérios do Canvas, um a um

O fecho do bloco 2 é o confronto formal: cada critério que o Canvas declarou, com o número que
o atende ou não. A regra é a que vale desde o Encontro 1 — **critério não atendido é registrado
como não atendido**, e reescrever o critério para caber no resultado é a forma mais comum de um
projeto falhar em silêncio.

Os critérios de negócio da seção 4 do Canvas — giro do estoque, ganho sobre custo de reforma,
preço realizado sobre anunciado — **não podem ser medidos aqui**, e isso não é uma falha: eles
medem o efeito de políticas que ainda não foram executadas. Ficam marcados como *pendentes de
implantação*, com a fase que os fecha. Marcá-los como atendidos seria inventar resultado; omiti-
los seria esconder que o projeto ainda não terminou.

In [ ]:
def veredito(atendido) -> str:
    """Traduz o veredito, tolerando `bool` do numpy — que não é `bool` do Python.

    A distinção não é preciosismo: `np.True_ is True` devolve `False`, e uma
    conferência escrita com `is` marcaria como pendente um critério atendido.
    """
    if atendido is None:
        return "pendente de implantação"
    return "atendido" if bool(atendido) else "não atendido"


def simbolo(atendido) -> str:
    return {"atendido": "✓", "não atendido": "✗", "pendente de implantação": "…"}[
        veredito(atendido)
    ]


criterios_canvas = pd.DataFrame(
    [
        {
            "origem": "Canvas §6 · técnico",
            "criterio": f"Silhueta ≥ {SILHUETA_MINIMA:.2f}",
            "medida": f"{SILHUETA_ESCOLHIDO:.4f}",
            "atendido": SILHUETA_ESCOLHIDO >= SILHUETA_MINIMA,
        },
        {
            "origem": "Canvas §6 · técnico",
            "criterio": f"Número de grupos entre {K_MINIMO} e {K_MAXIMO}",
            "medida": f"{K_ESCOLHIDO} grupos",
            "atendido": K_MINIMO <= K_ESCOLHIDO <= K_MAXIMO,
        },
        {
            "origem": "Canvas §6 · técnico",
            "criterio": f"Menor segmento ≥ {100 * MENOR_SEGMENTO_MINIMO:.0f}% da carteira",
            "medida": f"{100 * np.bincount(ROTULOS).min() / LINHAS:.1f}%",
            "atendido": (np.bincount(ROTULOS).min() / LINHAS) >= MENOR_SEGMENTO_MINIMO,
        },
        {
            "origem": "Canvas §6 · técnico",
            "criterio": f"Estabilidade sob reamostragem de 80%: ARI ≥ {ARI_MINIMO:.2f}",
            "medida": f"médio {manifesto_modelagem['ari_reamostragem_medio']:.4f} · "
                      f"mínimo {manifesto_modelagem['ari_reamostragem_minimo']:.4f}",
            "atendido": manifesto_modelagem["ari_reamostragem_minimo"] >= ARI_MINIMO,
        },
        {
            "origem": "Canvas §6 · técnico",
            "criterio": f"ARI contra quartis de preço < {ARI_PRECO_MAXIMO:.2f}",
            "medida": f"{ARI_QUARTIS:.4f}",
            "atendido": CRITERIO_PENDENTE_ATENDIDO,
        },
        {
            "origem": "Canvas §6 · técnico",
            "criterio": f"Ruído ≤ {100 * CRITERIOS['ruido_maximo']:.0f}% da carteira",
            "medida": "0% — o modelo escolhido atribui grupo a todo imóvel",
            "atendido": True,
        },
        {
            "origem": "Canvas §11 · ação",
            "criterio": "Cada linha admite política de preço, teto de reforma, canal e métrica",
            "medida": f"{K_ESCOLHIDO} linhas com perfil, valor e geografia distintos",
            "atendido": True,
        },
        {
            "origem": "Canvas §4 · negócio",
            "criterio": "100% do estoque com linha de produto atribuída em 30 dias",
            "medida": f"{LINHAS} de {LINHAS} imóveis classificados; o prazo depende da fase 6",
            "atendido": None,
        },
        {
            "origem": "Canvas §4 · negócio",
            "criterio": "Ganho de preço sobre custo de reforma ≥ 1,8× no segmento de maior retorno",
            "medida": "exige reformas executadas e revendas concluídas",
            "atendido": None,
        },
        {
            "origem": "Canvas §4 · negócio",
            "criterio": "Giro do segmento de entrada: de 90 para 60 dias",
            "medida": "exige a política em operação e um período de observação",
            "atendido": None,
        },
        {
            "origem": "Canvas §4 · negócio",
            "criterio": "Preço realizado ÷ anunciado ≥ 0,97 no alto padrão",
            "medida": "a base não traz preço anunciado — exige captura na fase 6",
            "atendido": None,
        },
        {
            "origem": "Canvas §4 · negócio",
            "criterio": "100% do estoque herdado com destino definido em 12 meses",
            "medida": "exige a decisão unidade a unidade prevista para o segmento 0",
            "atendido": None,
        },
    ]
)
# Normaliza para `bool` do Python (ou None) antes de qualquer leitura.
criterios_canvas["atendido"] = criterios_canvas.atendido.map(
    lambda valor: None if valor is None else bool(valor)
)
criterios_canvas["veredito"] = criterios_canvas.atendido.map(veredito)

CRITERIOS_TECNICOS = criterios_canvas.query("atendido.notna()")
TODOS_TECNICOS_ATENDIDOS = bool(CRITERIOS_TECNICOS.atendido.all())
PENDENTES_DE_IMPLANTACAO = int(criterios_canvas.atendido.isna().sum())

fig, ax = plt.subplots(figsize=(13, 2.6))
codigo = criterios_canvas.atendido.map({True: 1.0, False: 0.0}).fillna(0.5)
sns.heatmap(
    codigo.to_frame("veredito").T,
    annot=np.array([[simbolo(a) for a in criterios_canvas.atendido]]),
    fmt="", cmap=plt.matplotlib.colors.ListedColormap([ALERTA, REFERENCIA, APROVADO]),
    vmin=0, vmax=1, cbar=False, linewidths=1.4, linecolor="white",
    annot_kws={"fontsize": 17, "color": "white", "fontweight": "bold"},
    xticklabels=[f"{c[:40]}…" if len(c) > 40 else c for c in criterios_canvas.criterio], ax=ax,
)
ax.set_yticks([])
ax.tick_params(axis="x", rotation=38)
for rotulo, origem in zip(ax.get_xticklabels(), criterios_canvas.origem):
    rotulo.set_horizontalalignment("right")
    rotulo.set_fontsize(7.5)
    rotulo.set_color(config.AZUL_PROFUNDO if "técnico" in origem else config.CINZA)
ax.set_title(f"Os {len(criterios_canvas)} critérios do Canvas — "
             f"{int(CRITERIOS_TECNICOS.atendido.sum())} atendidos (verde), "
             f"{PENDENTES_DE_IMPLANTACAO} pendentes de implantação (amarelo)", pad=12)

salvar(fig, "criterios-do-canvas", criterios_canvas)
criterios_canvas[["origem", "criterio", "medida", "veredito"]]

## 2.6 · As perguntas SMART, respondidas de novo

A seção 9 do Canvas registra cinco perguntas orientadas a dados, com respostas publicadas. Elas
foram respondidas sobre a base anterior — 2.925 imóveis, 21 variáveis, quatro segmentos — e a
cadeia de notebooks 00 a 04 refez o percurso sobre uma carteira depurada de 2.350 imóveis e 17
variáveis.

Refazer as respostas é obrigação desta fase, não zelo: uma pergunta SMART com resposta
desatualizada é pior do que uma pergunta sem resposta, porque parece verificada.

In [ ]:
perguntas_smart = pd.DataFrame(
    [
        {
            "n": 1,
            "pergunta": "Quantos grupos distintos emergem, com silhueta ≥ 0,15 e no "
                        "máximo 5 grupos?",
            "resposta_publicada": "4 (silhueta 0,1775; base de 2.925 imóveis)",
            "resposta_medida": f"{K_ESCOLHIDO} (silhueta {SILHUETA_ESCOLHIDO:.4f}; base de "
                               f"{pt(LINHAS)} imóveis)",
            "situacao": "resposta mudou",
        },
        {
            "n": 2,
            "pergunta": "Qual grupo concentra mais de 40% da receita transacionada?",
            "resposta_publicada": "S4 · Novo de alto padrão, com 49,5%",
            "resposta_medida": f"Segmento {SEGMENTO_LIDER}, com {RECEITA_LIDER:.1f}%",
            "situacao": "resposta mudou de rótulo, conclusão preservada",
        },
        {
            "n": 3,
            "pergunta": "Que combinações de até três atributos ocorrem juntas com suporte ≥ 10% "
                        "e lift ≥ 1,30?",
            "resposta_publicada": "256 regras após poda; 252 diferenciais, cobrindo 4 segmentos",
            "resposta_medida": "não refeita — a cadeia 00–04 não contém a etapa de regras",
            "situacao": "sem resposta na cadeia atual",
        },
        {
            "n": 4,
            "pergunta": "Que percentual dos imóveis é classificado como ruído por um DBSCAN "
                        "com minPts = 2 × d?",
            "resposta_publicada": "1,81% (53 imóveis)",
            "resposta_medida": "a fase 4 varreu eps e não encontrou configuração com ruído "
                               "dentro do teto e número de grupos na faixa",
            "situacao": "resposta mudou",
        },
        {
            "n": 5,
            "pergunta": "A segmentação é distinguível de uma faixa de preço (ARI < 0,40)?",
            "resposta_publicada": "sim — ARI de 0,3112",
            "resposta_medida": f"sim — ARI de {ARI_QUARTIS:.4f}",
            "situacao": "conclusão confirmada, número atualizado",
        },
    ]
).set_index("n")

io.salvar_tabela(perguntas_smart.reset_index(), SECAO, "perguntas-smart", indice=False)

registrar(
    "2 · Resultados",
    "Três das cinco perguntas SMART do Canvas têm resposta diferente da publicada",
    "Canvas §9 · perguntas orientadas a dados",
    "As respostas publicadas foram medidas sobre a base anterior, de 2.925 imóveis e 21 "
    "variáveis. A cadeia atual roda sobre 2.350 imóveis depurados e 17 variáveis: não é o "
    "mesmo experimento, e a resposta não tem por que ser a mesma.",
    f"pergunta 1: 4 → {K_ESCOLHIDO} grupos · pergunta 4: 1,81% de ruído → nenhuma "
    f"configuração de DBSCAN admissível · pergunta 3: sem resposta na cadeia atual",
    acao="divergência registrada",
)
perguntas_smart

# 3 · Revisar o processo

*Review process* — a segunda tarefa da fase 5, e a que mais se pula. A tentação é entender a
revisão como uma retrospectiva de equipe: o que correu bem, o que correu mal, o que aprendemos.
O CRISP-DM pede outra coisa, e mais dura:

> Antes de prosseguir para a implantação, é preciso **rever o trabalho realizado**, verificar
> se algum fator importante foi negligenciado e se o modelo foi construído corretamente.

Ou seja: procurar o erro que passou. Uma revisão que não encontra nada não é uma revisão bem
sucedida — é uma revisão que não procurou.

Quatro exames, e três deles são mecânicos de propósito. Auditoria que depende de alguém lembrar
de conferir é auditoria que falha na semana em que a pessoa está de férias.

| Exame | Pergunta | Como se responde |
|:---|:---|:---|
| **3.1 · A cadeia** | Os artefatos são consistentes entre si? | Contagens e esquemas refeitos a partir dos manifestos |
| **3.2 · Vazamento** | O rótulo entrou em algum lugar? | Varredura das colunas de todo artefato que alimentou algoritmo |
| **3.3 · Reprodutibilidade** | Rodar de novo devolve o mesmo? | Reexecução do modelo a partir da semente, comparada rótulo a rótulo |
| **3.4 · Divergências** | O que o repositório afirma bate com o que foi medido? | Varredura dos documentos publicados contra as medidas desta cadeia |

## 3.1 · A cadeia de artefatos, refeita

Cada fase gravou um manifesto com o que entrou, o que saiu e o que foi decidido. A revisão liga
as pontas: a saída de cada fase tem de ser exatamente a entrada da seguinte, sem imóvel
aparecendo ou sumindo no caminho.

In [ ]:
manifesto_carga = io.ler_json(config.DATA_RAW / "manifesto_carga.json")
manifesto_curadoria = io.ler_json(config.DATA_PROCESSED / "manifesto_curadoria.json")

cadeia = pd.DataFrame(
    [
        {
            "fase": "2 · Carga",
            "artefato": "data/raw/ames_housing.parquet",
            "imoveis": int(manifesto_carga["linhas"]),
            "colunas": int(manifesto_carga["colunas"]),
            "notebook": "00 · dicionário de dados",
        },
        {
            "fase": "2 · Curadoria",
            "artefato": "data/processed/imoveis_curados.parquet",
            "imoveis": int(manifesto_curadoria["linhas_saida"]),
            "colunas": int(curada.shape[1]),
            "notebook": "02 · ajustes nos dados",
        },
        {
            "fase": "3 · Preparação",
            "artefato": "data/processed/base_modelagem.parquet",
            "imoveis": int(manifesto_preparacao["linhas"]),
            "colunas": int(manifesto_preparacao["colunas"]),
            "notebook": "03 · preparação",
        },
        {
            "fase": "4 · Modelagem",
            "artefato": "data/processed/base_segmentada.parquet",
            "imoveis": int(len(segmentada)),
            "colunas": int(segmentada.shape[1]),
            "notebook": "04 · modelagem",
        },
    ]
)
cadeia["variacao"] = cadeia.imoveis.diff().fillna(0).astype(int)

# As emendas entre fases: a saída de uma tem de ser a entrada da seguinte.
emendas = pd.DataFrame(
    [
        {
            "emenda": "carga → curadoria",
            "esperado": int(manifesto_carga["linhas"]),
            "observado": int(manifesto_curadoria["linhas_entrada"]),
        },
        {
            "emenda": "curadoria → preparação",
            "esperado": int(manifesto_curadoria["linhas_saida"]),
            "observado": int(manifesto_preparacao["linhas"]),
        },
        {
            "emenda": "preparação → modelagem",
            "esperado": int(manifesto_preparacao["linhas"]),
            "observado": int(sum(manifesto_modelagem["tamanhos"])),
        },
        {
            "emenda": "modelagem → avaliação",
            "esperado": int(sum(manifesto_modelagem["tamanhos"])),
            "observado": int(len(curada)),
        },
    ]
)
emendas["confere"] = emendas.esperado == emendas.observado
CADEIA_INTEGRA = bool(emendas.confere.all())

fig, eixos = plt.subplots(1, 2, figsize=(16, 4.8), width_ratios=[1.5, 1],
                          constrained_layout=True)

cores = [BARRA, BARRA, BARRA, APROVADO]
eixos[0].barh(cadeia.fase, cadeia.imoveis, color=cores)
eixos[0].bar_label(
    eixos[0].containers[0],
    labels=[f"{pt(n)} imóveis × {c} colunas" + (f"   ({v:+d})" if v else "")
            for n, c, v in zip(cadeia.imoveis, cadeia.colunas, cadeia.variacao)],
    padding=4, fontsize=9,
)
eixos[0].invert_yaxis()
eixos[0].set_xlim(0, cadeia.imoveis.max() * 1.45)
eixos[0].set_xlabel("Imóveis no artefato")
eixos[0].set_title("A cadeia de artefatos — a curadoria é o único ponto em que a carteira encolhe")

sns.heatmap(
    emendas.confere.to_frame("confere").T.astype(int),
    annot=np.array([["✓" if ok else "✗" for ok in emendas.confere]]), fmt="",
    cmap=plt.matplotlib.colors.ListedColormap([ALERTA, APROVADO]), vmin=0, vmax=1, cbar=False,
    linewidths=1.4, linecolor="white",
    annot_kws={"fontsize": 16, "color": "white", "fontweight": "bold"},
    xticklabels=emendas.emenda, ax=eixos[1],
)
eixos[1].set_yticks([])
eixos[1].tick_params(axis="x", rotation=28)
for rotulo in eixos[1].get_xticklabels():
    rotulo.set_horizontalalignment("right")
    rotulo.set_fontsize(8.5)
eixos[1].set_title("As emendas entre fases")

salvar(fig, "auditoria-da-cadeia", cadeia)
io.salvar_tabela(emendas, SECAO, "emendas-entre-fases", indice=False)
cadeia

## 3.2 · A auditoria de vazamento

O Canvas classifica o uso do rótulo reservado como risco **metodológico eliminatório** (seção
10). A defesa foi construída em camadas: a fase 3 gravou o rótulo fora do arquivo de modelagem,
a fase 4 conferiu na entrada, e este notebook confere de novo.

A auditoria abaixo não repete a conferência de um arquivo só. Ela varre **todo artefato que
alimentou ou pode alimentar algoritmo** — a base de modelagem, a base segmentada, a lista de
variáveis do modelo gravado e o pipeline serializado — procurando qualquer das três colunas
reservadas.

Redundância aqui é o ponto, não desperdício: para o erro passar, todas as camadas precisam
falhar ao mesmo tempo.

In [ ]:
# As variáveis que o pipeline gravado realmente viu, lidas do objeto e não do manifesto.
try:
    vistas_pelo_pipeline = set(pacote["pipeline"].named_steps["preparo"].feature_names_in_)
except AttributeError:
    vistas_pelo_pipeline = set(pacote["espaco"])

auditoria = pd.DataFrame(
    [
        {
            "artefato": "data/processed/base_modelagem.parquet",
            "papel": "entrada de todo algoritmo",
            "reservadas_encontradas": sorted(PROIBIDAS & set(base.columns)),
        },
        {
            "artefato": "data/processed/base_segmentada.parquet",
            "papel": "saída da fase 4",
            "reservadas_encontradas": sorted(PROIBIDAS & set(segmentada.columns)),
        },
        {
            "artefato": "manifesto_modelagem.json · variaveis",
            "papel": "declaração do espaço do modelo",
            "reservadas_encontradas": sorted(PROIBIDAS & set(manifesto_modelagem["variaveis"])),
        },
        {
            "artefato": "models/modelo-segmentacao.joblib",
            "papel": "o objeto que a fase 6 vai executar",
            "reservadas_encontradas": sorted(PROIBIDAS & vistas_pelo_pipeline),
        },
        {
            "artefato": "models/preprocessador-euclidiano.joblib",
            "papel": "receita de pré-processamento",
            "reservadas_encontradas": sorted(PROIBIDAS & set(ESPACO_EUCLIDIANO)),
        },
    ]
)
auditoria["limpo"] = auditoria.reservadas_encontradas.map(len) == 0
SEM_VAZAMENTO = bool(auditoria.limpo.all())

# Reprodutibilidade: o mesmo espaço e a mesma semente devolvem a mesma partição?
reexecutado = pers.ordenar_canonicamente(
    base, KMeans(n_clusters=K_ESCOLHIDO, init="k-means++", n_init=20,
                 random_state=SEMENTE).fit_predict(EUCLIDIANO)
)
ARI_REEXECUCAO = round(float(adjusted_rand_score(reexecutado, ROTULOS)), 4)
IDENTICO_NA_REEXECUCAO = bool((reexecutado == ROTULOS).all())

io.salvar_tabela(auditoria.assign(
    reservadas_encontradas=auditoria.reservadas_encontradas.map(lambda c: ", ".join(c) or "—")
), SECAO, "auditoria-de-vazamento", indice=False)

print(f"Artefatos auditados: {len(auditoria)} · sem vazamento: {SEM_VAZAMENTO}")
print(f"Reexecução do modelo com a semente {SEMENTE}: rótulos idênticos = "
      f"{IDENTICO_NA_REEXECUCAO} (ARI {ARI_REEXECUCAO:.4f})")
print(f"Modelo gravado reproduz a partição avaliada: {REPRODUZ} (ARI {ARI_REPRODUCAO:.4f})")
auditoria

In [ ]:
registrar(
    "3 · Processo",
    "Nenhuma coluna reservada alcançou artefato que alimenta algoritmo",
    f"{len(auditoria)} artefatos auditados",
    "O Canvas classifica o uso do rótulo reservado como risco metodológico eliminatório. A "
    "defesa é redundante de propósito: a fase 3 grava fora, a fase 4 confere na entrada e esta "
    "fase confere de novo, inclusive dentro do pipeline serializado.",
    f"0 ocorrências de {', '.join(sorted(PROIBIDAS))} em {len(auditoria)} artefatos",
    acao="decisão mantida",
)
registrar(
    "3 · Processo",
    "A segmentação é reproduzível a partir da semente e do arquivo",
    "Reprodutibilidade da fase 4",
    "Reexecutar o K-Means com a semente do projeto sobre o espaço reconstruído devolve a mesma "
    "atribuição, imóvel a imóvel — e o modelo serializado devolve a mesma partição que foi "
    "avaliada. Sem isso, a avaliação seria sobre um objeto que a fase 6 não receberia.",
    f"ARI da reexecução: {ARI_REEXECUCAO:.4f} · ARI do modelo gravado: {ARI_REPRODUCAO:.4f} · "
    f"cadeia de artefatos íntegra: {CADEIA_INTEGRA}",
    acao="decisão mantida",
)

## 3.3 · O que o repositório afirma, e o que a medida diz

Este é o exame que encontra alguma coisa — e encontra bastante.

A documentação publicada em `docs/`, o `README.md` e o catálogo de personas em
`src/model/personas.py` descrevem o **resultado anterior**: 2.925 imóveis, quatro linhas de
produto, persona S4 no alto padrão. A cadeia de notebooks 00 a 04 foi reconstruída sobre uma
carteira depurada de 2.350 imóveis e 17 variáveis, e devolveu três segmentos.

As duas coisas não são compatíveis, e a divergência não é erro de ninguém: é o rastro normal de
um projeto refeito com critério mais estrito. O que seria erro é **deixá-la em silêncio** — um
site que afirma quatro segmentos enquanto o modelo em produção entrega três é um site que vai
ser usado para tomar decisão errada.

A varredura abaixo é mecânica: procura nos documentos publicados os padrões que só fazem
sentido no resultado anterior, e conta onde eles ainda aparecem.

In [ ]:
import re

PADROES_DESATUALIZADOS = {
    "carteira de 2.925 imóveis": r"2\.925",
    "quatro segmentos, linhas ou personas": r"quatro (segmentos|linhas|personas)",
    "k = 4": r"\bk\s*=\s*4\b",
    "persona S4": r"\bS4\b",
}

alvos = sorted(
    [caminho for caminho in config.DOCS.rglob("*.md")]
    + [RAIZ / "README.md", RAIZ / "mkdocs.yml"]
)

linhas_varredura = []
for rotulo_padrao, expressao in PADROES_DESATUALIZADOS.items():
    arquivos, ocorrencias = [], 0
    for caminho in alvos:
        if not caminho.exists():
            continue
        achados = len(re.findall(expressao, caminho.read_text(encoding="utf-8", errors="ignore"),
                                 flags=re.IGNORECASE))
        if achados:
            arquivos.append(caminho.relative_to(RAIZ).as_posix())
            ocorrencias += achados
    linhas_varredura.append(
        {
            "afirmacao_desatualizada": rotulo_padrao,
            "ocorrencias": ocorrencias,
            "arquivos": len(arquivos),
            "onde": ", ".join(arquivos[:4]) + (" …" if len(arquivos) > 4 else ""),
        }
    )
varredura_documental = pd.DataFrame(linhas_varredura)
OCORRENCIAS_DESATUALIZADAS = int(varredura_documental.ocorrencias.sum())

divergencias = pd.DataFrame(
    [
        {
            "onde": "docs/ · README.md · mkdocs.yml",
            "o_que_afirma": "Carteira de 2.925 imóveis, quatro linhas de produto, persona S4",
            "o_que_a_medida_diz": f"{pt(LINHAS)} imóveis curados, {K_ESCOLHIDO} linhas de produto",
            "acao": "Reescrever as páginas de modelagem, avaliação e implantação sobre "
                    "a cadeia 00–04",
            "gravidade": "alta",
        },
        {
            "onde": "src/model/personas.py",
            "o_que_afirma": f"Catálogo com {len(pers.PERSONAS)} personas",
            "o_que_a_medida_diz": f"O modelo em produção tem {K_ESCOLHIDO} segmentos",
            "acao": f"Reescrever o catálogo para {K_ESCOLHIDO} personas a partir do perfil medido "
                    f"na fase 4, e travar com teste que compare len(PERSONAS) ao k do manifesto",
            "gravidade": "alta",
        },
        {
            "onde": "docs/pbl/canvas-do-problema.md §9",
            "o_que_afirma": "Cinco perguntas SMART com respostas da base anterior",
            "o_que_a_medida_diz": "Três das cinco mudaram de resposta; uma ficou sem resposta",
            "acao": "Atualizar as respostas e registrar a revisão na tabela de revisões do Canvas",
            "gravidade": "alta",
        },
        {
            "onde": "docs/pbl/canvas-do-problema.md §5",
            "o_que_afirma": "Regras de associação como validação independente e "
                             "anomalia como subproduto",
            "o_que_a_medida_diz": "A cadeia 00–04 não contém nenhuma das duas etapas",
            "acao": "Reconstruir a etapa de regras sobre a base curada, ou revisar a meta de "
                    "mineração no Canvas — as duas saídas são aceitáveis; o silêncio não é",
            "gravidade": "média",
        },
        {
            "onde": "src/pipeline/avaliacao.py",
            "o_que_afirma": "Consome `codigo_segmento`, `fora_de_padrao_dbscan` e "
                             "tabelas de associação",
            "o_que_a_medida_diz": "Nenhum desses artefatos é produzido pela cadeia atual",
            "acao": "Reescrever o módulo sobre os artefatos da cadeia 00–05 ou removê-lo do pacote",
            "gravidade": "média",
        },
    ]
)

fig, ax = plt.subplots(figsize=(13, 4.4))
ax.barh(varredura_documental.afirmacao_desatualizada, varredura_documental.ocorrencias,
        color=ALERTA)
ax.bar_label(
    ax.containers[0],
    labels=[f"{o} ocorrências em {a} arquivos"
            for o, a in zip(varredura_documental.ocorrencias, varredura_documental.arquivos)],
    padding=4, fontsize=9,
)
ax.invert_yaxis()
ax.set_xlim(0, varredura_documental.ocorrencias.max() * 1.5)
ax.set_xlabel("Ocorrências nos documentos publicados")
ax.set_title(f"O que o repositório ainda afirma sobre o resultado anterior — "
             f"{OCORRENCIAS_DESATUALIZADAS} ocorrências a revisar")

salvar(fig, "divergencias-documentais", varredura_documental)
io.salvar_tabela(divergencias, SECAO, "divergencias-a-resolver", indice=False)
divergencias

In [ ]:
registrar(
    "3 · Processo",
    "A documentação publicada descreve o resultado anterior, não o modelo em produção",
    "docs/, README.md, mkdocs.yml e src/model/personas.py",
    "A cadeia 00–04 foi reconstruída sobre uma carteira depurada e devolveu três segmentos; a "
    "documentação continua descrevendo quatro sobre 2.925 imóveis. Um site que afirma quatro "
    "linhas enquanto o modelo entrega três será usado para decidir errado.",
    f"{OCORRENCIAS_DESATUALIZADAS} ocorrências desatualizadas em documentos publicados · "
    f"catálogo com {len(pers.PERSONAS)} personas contra k = {K_ESCOLHIDO} no manifesto",
    acao="divergência registrada",
)
registrar(
    "3 · Processo",
    "Duas etapas prometidas na meta de mineração não existem na cadeia atual",
    "Canvas §5 · regras de associação e detecção de anomalia",
    "O Canvas define regras de associação como validação independente e anomalia como "
    "subproduto. A cadeia 00–04 não as contém. Reconstruí-las ou revisar a meta são saídas "
    "aceitáveis; deixar a promessa no Canvas sem entrega correspondente não é.",
    "0 de 2 etapas presentes na cadeia 00–04 · pergunta SMART 3 sem resposta",
    acao="divergência registrada",
)

## 3.4 · O balanço do processo

O que a revisão encontrou, separado entre o que sustentou e o que não sustentou.

**Sustentou:**

* **a ordem entre critério e resultado.** Os critérios de aceitação da fase 4 estavam escritos
  antes de qualquer algoritmo rodar, e este notebook os leu **do manifesto**, não os redigitou.
  É o que permitiu que o bloco 1 varresse 126 configurações sem que a régua se mexesse;
* **a reserva física do rótulo.** Foi ela que tornou possível o exame 2.1 — o critério de
  independência do preço só significa alguma coisa porque o preço estava comprovadamente fora.
  Se `preco_venda` estivesse na base de modelagem e alguém apenas tivesse "tomado cuidado", o
  ARI de 0,33 não provaria nada;
* **a decisão da fase 3 de gravar a receita crua.** Sem ela, as reamostras do bloco 1 teriam
  reaproveitado médias calculadas sobre a carteira inteira, e a estabilidade sairia
  otimista;
* **a ordenação canônica dos segmentos.** Sem ela, "segmento 2" mudaria de identidade a cada
  execução e toda frase de negócio deste notebook envelheceria em uma rodada.

**Não sustentou:**

* **a documentação publicada**, que descreve outro experimento — a maior dívida aberta;
* **o catálogo de personas**, que tem mais personas do que o modelo tem segmentos. É o tipo de
  inconsistência que um teste automatizado pega e nenhuma revisão manual pega sempre;
* **a promessa de regras de associação e anomalia**, que ficou no Canvas sem entrega
  correspondente na cadeia atual.

**O que se faria diferente**, se a fase 3 recomeçasse: fixar o piso de silhueta **em função do
número de grupos**, e não como valor único. O bloco 1 mostrou que a silhueta cai
sistematicamente quando *k* sobe — é propriedade da métrica, não da carteira. Um piso único
favorece *k* pequeno por construção, e foi ele que reprovou *k* = 4 por uma margem estreita. Um
piso por faixa de *k*, declarado no Canvas antes de qualquer varredura, seria mais honesto com
o que a métrica realmente mede. Fica registrado como revisão de método — para a próxima
execução, não para esta.

# 4 · Determinar as próximas etapas

*Determine next steps* — a última tarefa da fase 5, e a que o CRISP-DM define como uma decisão
de três saídas, não de duas: **prosseguir para a implantação**, **voltar para uma fase
anterior** ou **iniciar um novo projeto**.

A decisão sai do que os blocos 2 e 3 mediram, e não de disposição para seguir.

In [ ]:
FALHAS_TECNICAS = int((~CRITERIOS_TECNICOS.atendido.astype(bool)).sum())
DIVERGENCIAS_ALTAS = int((divergencias.gravidade == "alta").sum())

decisao = pd.Series(
    {
        "critérios técnicos do Canvas": f"{int(CRITERIOS_TECNICOS.atendido.sum())} de "
                                        f"{len(CRITERIOS_TECNICOS)} atendidos",
        "critério pendente da fase 4": f"atendido — ARI {ARI_QUARTIS:.4f} contra o teto "
                                       f"{ARI_PRECO_MAXIMO:.2f}",
        "validação externa": f"V de Cramér {V_CRAMER:.3f} contra `bairro`",
        "vazamento do rótulo": "nenhum em " f"{len(auditoria)} artefatos",
        "reprodutibilidade": f"ARI {ARI_REEXECUCAO:.4f} na reexecução",
        "cadeia de artefatos": "íntegra" if CADEIA_INTEGRA else "inconsistente",
        "divergências de gravidade alta": f"{DIVERGENCIAS_ALTAS} abertas",
    },
    name="o que sustenta a decisão",
)

PROSSEGUE = bool(TODOS_TECNICOS_ATENDIDOS and SEM_VAZAMENTO and CADEIA_INTEGRA and REPRODUZ)
VEREDITO = (
    "prosseguir para a fase 6, com as divergências documentais como condição de publicação"
    if PROSSEGUE
    else "voltar para a fase anterior"
)
print(f"Decisão: {VEREDITO.upper()}")
decisao.to_frame()

## 4.1 · A decisão

> **Prosseguir para a fase 6 — implantação — com uma condição.**

O modelo sustenta os três exames que decidem: passa em todos os critérios técnicos do Canvas,
atende o critério de independência do preço que estava pendente e se sustenta contra uma
variável que nunca entrou no algoritmo. A cadeia de artefatos está íntegra, não há vazamento e
o resultado é reproduzível a partir da semente.

A condição é a dívida documental. **Nada do que a fase 6 publicar pode se apoiar nas páginas
atuais de `docs/`**, que descrevem outro experimento. Implantar um modelo de três segmentos com
documentação de quatro é criar, no primeiro dia de operação, exatamente o tipo de erro que este
projeto passou cinco fases evitando.

Não é uma ressalva formal: as três divergências de gravidade alta bloqueiam a publicação, e o
backlog abaixo as coloca antes de qualquer item de produto.

## 4.2 · O backlog da fase 6

Cada item tem um responsável natural, um critério que o fecha e uma ordem. Item sem critério de
conclusão não é tarefa, é intenção.

In [ ]:
proximas_etapas = pd.DataFrame(
    [
        {
            "ordem": 1,
            "etapa": f"Reescrever o catálogo de personas para {K_ESCOLHIDO} segmentos",
            "por_que": f"O catálogo tem {len(pers.PERSONAS)} personas e o modelo tem "
                       f"{K_ESCOLHIDO} segmentos: a fase 6 não pode nomear o que não existe",
            "onde": "src/model/personas.py",
            "criterio_de_conclusao": "len(PERSONAS) == k do manifesto, travado por teste "
                                     "automatizado, com ação e métrica por segmento",
            "bloqueia_publicacao": True,
        },
        {
            "ordem": 2,
            "etapa": "Atualizar a documentação publicada para a cadeia 00–05",
            "por_que": f"{OCORRENCIAS_DESATUALIZADAS} ocorrências descrevem o resultado anterior "
                       f"em páginas que a diretoria lê",
            "onde": "docs/modelagem.md, docs/avaliacao.md, docs/implementacao.md, "
                    "docs/index.md, README.md, mkdocs.yml",
            "criterio_de_conclusao": "a varredura documental deste notebook volta com zero "
                                     "ocorrências desatualizadas",
            "bloqueia_publicacao": True,
        },
        {
            "ordem": 3,
            "etapa": "Atualizar as respostas SMART e registrar a revisão do Canvas",
            "por_que": "Três das cinco respostas mudaram; uma ficou sem resposta na cadeia atual",
            "onde": "docs/pbl/canvas-do-problema.md §9 e tabela de revisões",
            "criterio_de_conclusao": "as cinco perguntas com resposta medida nesta cadeia e "
                                     "link para a evidência",
            "bloqueia_publicacao": True,
        },
        {
            "ordem": 4,
            "etapa": "Tratar a fronteira entre segmentos na regra de atribuição",
            "por_que": f"{IMOVEIS_NA_FRONTEIRA} imóveis estão em silhueta negativa, concentrados "
                       f"no segmento {SEGMENTO_FRAGIL}: a política erra justamente neles",
            "onde": "src/deployment/scoring.py",
            "criterio_de_conclusao": "a saída do scoring traz distância ao segundo centro "
                                     "mais próximo e marca o imóvel como fronteiriço, para "
                                     "revisão humana",
            "bloqueia_publicacao": False,
        },
        {
            "ordem": 5,
            "etapa": "Instrumentar as métricas de negócio do Canvas §4",
            "por_que": f"{PENDENTES_DE_IMPLANTACAO} critérios de negócio não podem ser medidos "
                       f"sem a política em operação",
            "onde": "src/deployment/monitoramento.py",
            "criterio_de_conclusao": "giro, ganho sobre custo de reforma e preço realizado "
                                     "÷ anunciado capturados por segmento, com linha de "
                                     "base registrada",
            "bloqueia_publicacao": False,
        },
        {
            "ordem": 6,
            "etapa": f"Levar ao Canvas a revisão do piso de silhueta e o candidato k = {K_REVISAO}",
            "por_que": f"k = {K_REVISAO} reprova por margem estreita e tem a distribuição de "
                       f"receita mais equilibrada; o piso único favorece k pequeno por construção",
            "onde": "docs/pbl/canvas-do-problema.md §6",
            "criterio_de_conclusao": "piso de silhueta declarado por faixa de k, decidido ANTES "
                                     "da próxima execução da cadeia",
            "bloqueia_publicacao": False,
        },
        {
            "ordem": 7,
            "etapa": "Decidir o destino da meta de mineração não entregue",
            "por_que": "O Canvas §5 promete regras de associação e detecção de anomalia; a "
                       "cadeia atual não tem nenhuma das duas",
            "onde": "src/model/associacao.py e Canvas §5",
            "criterio_de_conclusao": "ou a etapa de regras roda sobre a base curada, ou a meta "
                                     "de mineração é revista no Canvas — com a revisão registrada",
            "bloqueia_publicacao": False,
        },
        {
            "ordem": 8,
            "etapa": "Definir o gatilho de reciclagem do modelo",
            "por_que": "A carteira recebe imóveis novos, e a segmentação envelhece sem avisar",
            "onde": "src/deployment/monitoramento.py",
            "criterio_de_conclusao": "regra escrita: reajustar quando o ARI entre a partição "
                                     "vigente e a reajustada cair abaixo do piso de estabilidade "
                                     f"({ARI_MINIMO:.2f})",
            "bloqueia_publicacao": False,
        },
    ]
).set_index("ordem")

fig, ax = plt.subplots(figsize=(13, 5.6))
cores = [ALERTA if bloqueia else BARRA for bloqueia in proximas_etapas.bloqueia_publicacao]
posicoes = np.arange(len(proximas_etapas))
ax.barh(posicoes, [1] * len(proximas_etapas), color=cores, height=0.72)
for posicao, (ordem, linha) in zip(posicoes, proximas_etapas.iterrows()):
    ax.text(0.012, posicao, f"{ordem}. {linha.etapa}", va="center", ha="left", color="white",
            fontsize=9.5, fontweight="bold")
    ax.text(0.988, posicao, "bloqueia a publicação" if linha.bloqueia_publicacao else "fase 6",
            va="center", ha="right", color="white", fontsize=8.5)
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.invert_yaxis()
ax.grid(False)
for lado in ("top", "right", "bottom", "left"):
    ax.spines[lado].set_visible(False)
ax.set_title(f"O backlog da fase 6 — {int(proximas_etapas.bloqueia_publicacao.sum())} itens "
             f"bloqueiam a publicação (coral)")

salvar(fig, "proximas-etapas", proximas_etapas.reset_index())
proximas_etapas[["etapa", "onde", "criterio_de_conclusao", "bloqueia_publicacao"]]

## 4.3 · O que a fase 6 recebe

Nenhum modelo novo é gravado aqui, e a ausência é deliberada: a fase 5 **avalia**, não
reajusta. O `models/modelo-segmentacao.joblib` que a fase 6 vai executar é exatamente o objeto
que a fase 4 gravou — conferido no bloco 0 como reprodutor fiel da partição avaliada.

O que este notebook acrescenta é o **veredito**, com as medidas que o sustentam:

| Artefato | Conteúdo |
|:---|:---|
| `data/processed/manifesto_avaliacao.json` | O veredito, os critérios confrontados, o backlog e o que ficou pendente |
| `reports/tables/avaliacao/*.csv` | Toda tabela que originou uma figura, e as que não têm figura |
| `docs/imagens/avaliacao/*.png` | As figuras da fase, para o MkDocs |

In [ ]:
config.garantir_diretorios()

manifesto_avaliacao = {
    "origem": {
        "base_modelagem": BASE_MODELAGEM.relative_to(RAIZ).as_posix(),
        "base_segmentada": BASE_SEGMENTADA.relative_to(RAIZ).as_posix(),
        "modelo": MODELO.relative_to(RAIZ).as_posix(),
        "rotulo_reservado_aberto_de": config.BASE_CURADA.relative_to(RAIZ).as_posix(),
    },
    "modelo_avaliado": {
        "algoritmo": ALGORITMO,
        "k": K_ESCOLHIDO,
        "silhueta": SILHUETA_ESCOLHIDO,
        "reproduz_a_particao_avaliada": REPRODUZ,
        "ari_da_reexecucao": ARI_REEXECUCAO,
    },
    "varredura_de_k": {
        "configuracoes_de_k_por_algoritmo": int(len(grade)),
        "configuracoes_de_escala": int(len(escala)),
        "configuracoes_de_dimensao": int(len(reducao)),
        "total": int(len(grade) + len(escala) + len(reducao)),
        "melhor_silhueta_absoluta": {
            "configuracao": CAMPEAO_LITERAL,
            "silhueta": SILHUETA_MAXIMA,
            "menor_grupo": MENOR_GRUPO_CAMPEAO,
            "admissivel": False,
            "motivo": "menor grupo abaixo do piso de 2% da carteira",
        },
        "melhor_silhueta_admissivel": {
            "configuracao": f"{MELHOR_ADMISSIVEL.algoritmo}, k = {K_MELHOR_SILHUETA}",
            "silhueta": SILHUETA_MELHOR_ADMISSIVEL,
            "adotada": False,
            "motivo": "reproduz a política de duas faixas em vigor — ARI "
                      f"{independencia.loc[2, 'ari_contra_a_mediana']:.4f} contra o "
                      "corte pela mediana de preço",
        },
        "k_pela_regra_declarada": K_PELA_REGRA,
        "k_pela_silhueta": K_PELA_SILHUETA,
    },
    "avaliacao_de_negocio": {
        "ari_contra_quartis_de_preco": ARI_QUARTIS,
        "teto_do_canvas": ARI_PRECO_MAXIMO,
        "criterio_pendente_atendido": CRITERIO_PENDENTE_ATENDIDO,
        "ari_contra_o_corte_pela_mediana": ARI_MEDIANA,
        "v_de_cramer_contra_bairro": round(V_CRAMER, 4),
        "segmento_lider_em_receita": SEGMENTO_LIDER,
        "pct_receita_do_lider": RECEITA_LIDER,
        "segmento_de_fronteira": SEGMENTO_FRAGIL,
        "imoveis_em_silhueta_negativa": IMOVEIS_NA_FRONTEIRA,
    },
    "criterios_do_canvas": {
        "tecnicos_atendidos": int(CRITERIOS_TECNICOS.atendido.sum()),
        "tecnicos_totais": int(len(CRITERIOS_TECNICOS)),
        "de_negocio_pendentes_de_implantacao": PENDENTES_DE_IMPLANTACAO,
    },
    "revisao_do_processo": {
        "cadeia_integra": CADEIA_INTEGRA,
        "artefatos_auditados": int(len(auditoria)),
        "vazamento_encontrado": not SEM_VAZAMENTO,
        "ocorrencias_documentais_desatualizadas": OCORRENCIAS_DESATUALIZADAS,
        "divergencias_abertas": int(len(divergencias)),
        "divergencias_de_gravidade_alta": DIVERGENCIAS_ALTAS,
    },
    "veredito": VEREDITO,
    "prossegue_para_implantacao": PROSSEGUE,
    "itens_que_bloqueiam_a_publicacao": int(proximas_etapas.bloqueia_publicacao.sum()),
    "candidato_de_revisao": {
        "k": K_REVISAO,
        "silhueta": float(confronto.loc[K_REVISAO, "silhueta"]),
        "distancia_do_piso": round(folga_silhueta, 4),
        "decidir_em": "Canvas §6, antes da próxima execução da cadeia",
    },
    "semente": SEMENTE,
    "julgamentos": len(JULGAMENTOS),
}

registro = pd.DataFrame(JULGAMENTOS).set_index("n")
io.salvar_tabela(registro.reset_index(), SECAO, "registro-de-julgamentos", indice=False)
manifesto_avaliacao["julgamentos_por_acao"] = registro.acao.value_counts().to_dict()
io.salvar_json(manifesto_avaliacao, config.DATA_PROCESSED / "manifesto_avaliacao.json")

print("Manifesto gravado em data/processed/manifesto_avaliacao.json")
print(f"{len(registro)} julgamentos registrados")
print(f"Veredito: {VEREDITO}")

In [ ]:
registro

# Próximo passo

A fase 5 termina com um modelo **aprovado sob a ótica do negócio** e uma dívida documental
aberta que precisa ser paga antes de qualquer publicação.

O que ficou estabelecido, e que a fase 6 pode citar sem remedir:

* **a segmentação não é uma tabela de preço.** O critério que a fase 4 deixou pendente foi
  medido e atendido — e ele só significa alguma coisa porque o preço esteve comprovadamente
  fora de todo artefato que alimentou algoritmo;
* **os segmentos existem fora do espaço em que foram criados.** A associação com `bairro`, uma
  variável que nenhum algoritmo viu, é forte o bastante para que nenhum bairro se pareça com a
  carteira média;
* **a busca pela melhor silhueta foi feita, e o resultado dela sustenta a decisão da fase 4.**
  Das 126 configurações, a de maior silhueta separa um imóvel dos demais; a melhor admissível é
  *k* = 2, que é a política de duas faixas que o projeto foi contratado para substituir. A
  regra declarada antes do resultado escolheu melhor do que a métrica escolheria;
* **um em cada quatro imóveis do segmento do meio está perto da fronteira.** É o achado que a
  fase 6 precisa absorver no desenho da regra de atribuição, e não em mais uma rodada de
  modelagem.

A primeira tarefa da fase 6 não é de produto: é reescrever o catálogo de personas para o número
de segmentos que o modelo realmente tem. Enquanto `src/model/personas.py` descrever quatro
personas e o manifesto declarar três, qualquer aplicação construída em cima nomeia um segmento
que não existe — e o erro aparece na primeira tela que a diretoria abrir.